# YOLOv8m Cross-Year AF-MCSO: Competitive 320-Unit HPO

This notebook implements the final cross-year Adaptive Fidelity-Aware Modified Cuckoo Search (AF-MCSO) experiment for YOLOv8m.

HPO uses only source-year training and validation data. The target-year test split remains isolated until the best measured 40-epoch source-validation configuration has been frozen. The total HPO budget is fixed at 320 epoch-units, matching the single-year AF-MCSO comparison.

The notebook is prepared for GitHub review. `RUN_CROSS_YEAR_AF_MCSO` is `False` by default.


## Module 1: Cross-year direction, frozen protocol, budget, and portable paths

Defines the source and target year selectors, YOLOv8m runtime settings, focused 5D search space, AF-MCSO controls, Year2021 source-training subsampling rule, exact 320 epoch-unit HPO budget, seeds, and repository-relative output locations. No dataset is scanned in this module.


In [ ]:
from pathlib import Path
from datetime import datetime
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter, defaultdict

import csv
import json
import math
import os
import random
import shutil
import sys
import time
import uuid
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO


# ============================================================
# A. Cross-year direction
# ============================================================
# Change only these two values to reverse the experiment.
TRAIN_YEAR = "2021"
TEST_YEAR = "2022"

VALID_YEARS = {"2021", "2022"}
if TRAIN_YEAR not in VALID_YEARS or TEST_YEAR not in VALID_YEARS:
    raise ValueError("TRAIN_YEAR and TEST_YEAR must each be '2021' or '2022'.")
if TRAIN_YEAR == TEST_YEAR:
    raise ValueError("Cross-year evaluation requires TRAIN_YEAR != TEST_YEAR.")

TRAIN_DATASET_TAG = f"Year{TRAIN_YEAR}"
TEST_DATASET_TAG = f"Year{TEST_YEAR}"
CROSS_YEAR_TAG = f"Y{TRAIN_YEAR}_to_Y{TEST_YEAR}"


# ============================================================
# B. Portable repository paths
# ============================================================
REPO_ROOT = Path.cwd()
TRAIN_SOURCE_ROOT = REPO_ROOT / "Datasets" / TRAIN_DATASET_TAG
TEST_TARGET_ROOT = REPO_ROOT / "Datasets" / TEST_DATASET_TAG

PROJECT_ROOT = REPO_ROOT / "outputs" / f"YOLOv8m_CrossYear_AF_MCSO_{CROSS_YEAR_TAG}"
RUNS_ROOT = PROJECT_ROOT / "runs"
SPLITS_ROOT = PROJECT_ROOT / "splits"
STATS_ROOT = PROJECT_ROOT / "stats"
YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUNS_ROOT / "af_mcso20to40_320" / RUN_TIMESTAMP
HPO_RUNS_ROOT = RUN_ROOT / "hpo"
FINAL_TRAINING_ROOT = RUN_ROOT / "final40"
TARGET_TEST_EVAL_ROOT = RUN_ROOT / "target_test"
RUN_STATS_ROOT = RUN_ROOT / "stats"
RUN_PLOTS_ROOT = RUN_ROOT / "plots"


def ensure_output_directories() -> None:
    """Create experiment output folders only when the workflow is run."""
    for folder in [
        PROJECT_ROOT,
        RUNS_ROOT,
        SPLITS_ROOT,
        STATS_ROOT,
        YOLO_DATA_ROOT,
        RUN_ROOT,
        HPO_RUNS_ROOT,
        FINAL_TRAINING_ROOT,
        TARGET_TEST_EVAL_ROOT,
        RUN_STATS_ROOT,
        RUN_PLOTS_ROOT,
    ]:
        folder.mkdir(parents=True, exist_ok=True)


def portable_path(path: Path) -> str:
    """Return a repository-relative path when possible."""
    path = Path(path)
    try:
        return path.resolve().relative_to(REPO_ROOT.resolve()).as_posix()
    except ValueError:
        return path.name


def resolve_repo_path(path_value: Any) -> Path:
    """Resolve a stored portable path for runtime use."""
    path = Path(path_value)
    return path if path.is_absolute() else REPO_ROOT / path


def sanitize_exception_message(error: Exception) -> str:
    """Remove repository and home paths from a saved error message."""
    message = f"{type(error).__name__}: {error}"
    for value in [str(REPO_ROOT.resolve()), str(Path.home().resolve())]:
        if value:
            message = message.replace(value, "<local_path>")
            message = message.replace(value.replace("\\", "/"), "<local_path>")
    return message


# ============================================================
# C. Classes and label aliases
# ============================================================
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}


# ============================================================
# D. Split and seed protocol
# ============================================================
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

OPTIMIZATION_SPLIT_SEED = 42
OPTIMIZATION_TRAIN_SEED = 42
FINAL_EVALUATION_SEEDS = [42, 123, 999]

# Method-specific AF-MCSO seed retained from the validated single-year design.
MCSO_RANDOM_SEED = 9187


# ============================================================
# E. Year2021 source-training subsampling
# ============================================================
# Applied only when TRAIN_YEAR == "2021".
# Only the source TRAIN split is reduced to the target-year TRAIN-split size.
SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE = True
SUBSAMPLE_TARGET_MODE = "target_train_size"
SUBSAMPLE_SEED_OFFSET = 10000


# ============================================================
# F. Shared YOLO runtime protocol
# ============================================================
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"
CACHE_MODE = "disk"

LOW_FIDELITY_EPOCHS = 20
LOW_FIDELITY_WARMUP_EPOCHS = 1.5
LOW_FIDELITY_CLOSE_MOSAIC = 5

FULL_FIDELITY_EPOCHS = 40
FULL_FIDELITY_WARMUP_EPOCHS = 3.0
FULL_FIDELITY_CLOSE_MOSAIC = 10

FINAL_EPOCHS = 40
FINAL_WARMUP_EPOCHS = 3.0
FINAL_CLOSE_MOSAIC = 10


# ============================================================
# G. Frozen focused 5D HPO search space
# ============================================================
SEARCH_SPACE_VERSION = "Focused5D_CrossYear_AFMCSO20to40_320_mAP5095_v1"

SEARCH_SPACE = {
    "learning_rate": {"low": 0.0010, "high": 0.0100, "distribution": "log"},
    "lrf":           {"low": 0.0100, "high": 0.2000, "distribution": "log"},
    "momentum":      {"low": 0.8500, "high": 0.9700, "distribution": "linear"},
    "weight_decay":  {"low": 0.0001, "high": 0.0010, "distribution": "log"},
    "scale":         {"low": 0.2000, "high": 0.8000, "distribution": "linear"},
}

SEARCH_PARAMETER_ORDER = [
    "learning_rate",
    "lrf",
    "momentum",
    "weight_decay",
    "scale",
]

REFERENCE_UNTUNED_VALUES = {
    "learning_rate": 0.010,
    "lrf": 0.010,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

for parameter_name, reference_value in REFERENCE_UNTUNED_VALUES.items():
    spec = SEARCH_SPACE[parameter_name]
    if not float(spec["low"]) <= float(reference_value) <= float(spec["high"]):
        raise RuntimeError(
            f"Reference value {parameter_name}={reference_value} is outside "
            f"[{spec['low']}, {spec['high']}]."
        )


# ============================================================
# H. Fixed non-tuned training settings
# ============================================================
FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": CACHE_MODE,
    "cos_lr": False,
    "rect": False,
}


# ============================================================
# I. AF-MCSO search mechanics
# ============================================================
# These mechanics are deliberately unchanged from the validated single-year
# AF-MCSO experiment. Only the cross-year data routing is changed.
INITIAL_NESTS = 4
INITIAL_CONFIRMATIONS = 2
INITIAL_MAXIMIN_POOL = 1000

MCSO_GENERATIONS = 2
CANDIDATES_PER_GENERATION = 2
PROMOTIONS_PER_GENERATION = 1

ELITE_ARCHIVE_SIZE = 3
ARCHIVE_MIN_DISTANCE = 0.08

LEVY_BETA = 1.5
GENERATION_STEP_SCALES = [0.18, 0.10]
STAGNATION_TOLERANCE = 1e-4
STAGNATION_SCALE_MULTIPLIER = 1.75
OPPOSITION_DIRECTION_WEIGHT = 0.50
STAGNATION_OPPOSITION_WEIGHT = 0.70
IMMIGRANT_POOL_SIZE = 500
SIGNATURE_PRECISION = 12

# Adaptive-fidelity prediction estimates expected 40-epoch SOURCE-validation
# mAP50-95 from completed 20->40 SOURCE-validation pairs. It controls promotion
# only and never changes the final HPO objective.
AF_K_NEIGHBORS = 3
AF_MIN_HISTORY_PAIRS = 2
AF_DISTANCE_EPS = 1e-12


# ============================================================
# J. Exact competitive HPO budget
# ============================================================
# Stage 0:      4 x 20 + 2 x 40 = 160 epoch-units
# Generation 1: 2 x 20 + 1 x 40 =  80 epoch-units
# Generation 2: 2 x 20 + 1 x 40 =  80 epoch-units
# HPO total:                        320 epoch-units
INITIAL_STAGE_UNITS = (
    INITIAL_NESTS * LOW_FIDELITY_EPOCHS
    + INITIAL_CONFIRMATIONS * FULL_FIDELITY_EPOCHS
)
PER_GENERATION_UNITS = (
    CANDIDATES_PER_GENERATION * LOW_FIDELITY_EPOCHS
    + PROMOTIONS_PER_GENERATION * FULL_FIDELITY_EPOCHS
)
PLANNED_HPO_EPOCH_UNITS = INITIAL_STAGE_UNITS + MCSO_GENERATIONS * PER_GENERATION_UNITS

EXPECTED_LOW_FIDELITY_EVALUATIONS = INITIAL_NESTS + MCSO_GENERATIONS * CANDIDATES_PER_GENERATION
EXPECTED_FULL_FIDELITY_EVALUATIONS = INITIAL_CONFIRMATIONS + MCSO_GENERATIONS * PROMOTIONS_PER_GENERATION
EXPECTED_TOTAL_HPO_EVALUATIONS = EXPECTED_LOW_FIDELITY_EVALUATIONS + EXPECTED_FULL_FIDELITY_EVALUATIONS

FINAL_EVALUATION_EPOCH_UNITS = len(FINAL_EVALUATION_SEEDS) * FINAL_EPOCHS
PLANNED_FULL_WORKFLOW_EPOCH_UNITS = PLANNED_HPO_EPOCH_UNITS + FINAL_EVALUATION_EPOCH_UNITS

assert INITIAL_STAGE_UNITS == 160
assert PER_GENERATION_UNITS == 80
assert PLANNED_HPO_EPOCH_UNITS == 320
assert EXPECTED_LOW_FIDELITY_EVALUATIONS == 8
assert EXPECTED_FULL_FIDELITY_EVALUATIONS == 4
assert EXPECTED_TOTAL_HPO_EVALUATIONS == 12
assert FINAL_EVALUATION_EPOCH_UNITS == 120
assert PLANNED_FULL_WORKFLOW_EPOCH_UNITS == 440
assert len(GENERATION_STEP_SCALES) == MCSO_GENERATIONS

REQUIRE_FULL_HPO_BUDGET = True
REQUIRE_ALL_FINAL_RUNS = True
STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002


def get_fidelity_controls(budget_epochs: int) -> Tuple[float, int]:
    """Return the frozen 20- or 40-epoch training schedule."""
    budget_epochs = int(budget_epochs)
    if budget_epochs == LOW_FIDELITY_EPOCHS:
        return LOW_FIDELITY_WARMUP_EPOCHS, LOW_FIDELITY_CLOSE_MOSAIC
    if budget_epochs == FULL_FIDELITY_EPOCHS:
        return FULL_FIDELITY_WARMUP_EPOCHS, FULL_FIDELITY_CLOSE_MOSAIC
    raise ValueError(f"Unsupported AF-MCSO fidelity: {budget_epochs}")


print("=" * 100)
print("YOLOv8m CROSS-YEAR AF-MCSO : 20/40 : 320 HPO EPOCH-UNITS : mAP50-95")
print("=" * 100)
print("Direction:", CROSS_YEAR_TAG)
print("Source/train dataset:", portable_path(TRAIN_SOURCE_ROOT))
print("Target/test dataset:", portable_path(TEST_TARGET_ROOT))
print("HPO objective:", "SOURCE-year validation mAP50-95 only")
print("Target-year test used during HPO: NO")
print("Search space:", json.dumps(SEARCH_SPACE, indent=2))
print("MCSO seed:", MCSO_RANDOM_SEED)
print("Fidelities:", LOW_FIDELITY_EPOCHS, "to", FULL_FIDELITY_EPOCHS)
print("HPO budget:", PLANNED_HPO_EPOCH_UNITS, "epoch-units")
print("Expected HPO evaluations:", EXPECTED_TOTAL_HPO_EVALUATIONS)
print("Expected 20-epoch evaluations:", EXPECTED_LOW_FIDELITY_EVALUATIONS)
print("Expected 40-epoch evaluations:", EXPECTED_FULL_FIDELITY_EVALUATIONS)
print("Final evaluation budget:", FINAL_EVALUATION_EPOCH_UNITS, "epoch-units")
print("Full workflow training budget:", PLANNED_FULL_WORKFLOW_EPOCH_UNITS, "epoch-units")
print("Workers / cache:", WORKERS, "/", CACHE_MODE)
print("Patience:", PATIENCE, "(0 = EarlyStopping disabled)")
print("Optimization split/train seed:", OPTIMIZATION_SPLIT_SEED, "/", OPTIMIZATION_TRAIN_SEED)
print("Final seeds:", FINAL_EVALUATION_SEEDS)
print("Year2021-source subsampling enabled:", SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE)
print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output root:", portable_path(RUN_ROOT))
print("=" * 100)



## Module 2: Annotation parsing and normalized record helpers

Reads Pascal VOC XML and common JSON annotations, normalizes the known class aliases, reads image dimensions when required, and clips bounding boxes to valid image boundaries.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def normalize_label(raw_label: str) -> str:
    """Normalize one class label using the fixed alias table."""
    cleaned = str(raw_label).strip()
    return LABEL_ALIASES.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image width and height, failing clearly for unreadable files."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}\n{error}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML annotation."""
    root = ET.parse(xml_path).getroot()
    xml_size = None
    size_node = root.find("size")

    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            xml_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []
    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")
        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]
        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return xml_size, objects


def extract_via_label(attributes: Dict[str, Any]) -> Optional[str]:
    """Extract a label from common VIA region-attribute layouts."""
    preferred_keys = [
        "label", "class", "class_name", "name", "species", "type", "weed", "category"
    ]

    for key in preferred_keys:
        value = attributes.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_label(value)
        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key)
            if len(value) == 1:
                return normalize_label(next(iter(value.keys())))

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value)

    return None


def parse_json_annotation(json_path: Path) -> List[Dict[str, Any]]:
    """Parse LabelMe-style or VIA-style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    # LabelMe
    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""))
            points = shape.get("points", [])
            if not label or len(points) < 2:
                continue

            xs = [float(point[0]) for point in points]
            ys = [float(point[1]) for point in points]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({"label": label, "bbox_xyxy": [x1, y1, x2, y2]})
        return objects

    # VIA
    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue

        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            regions = [regions] if "shape_attributes" in regions else list(regions.values())
        if not isinstance(regions, list):
            continue

        for region in regions:
            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}
            if shape.get("name") != "rect":
                continue

            values = [shape.get("x"), shape.get("y"), shape.get("width"), shape.get("height")]
            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip one xyxy box to image boundaries."""
    x1, y1, x2, y2 = box
    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None
    return [x1, y1, x2, y2]

## Module 3: Build and audit source and target records

Builds clean records for both years only when the experiment runs. It keeps the eight target classes, removes invalid boxes, checks class coverage, saves compact audit summaries, and creates one source-versus-target class distribution figure.


In [ ]:
def build_clean_records(
    source_root: Path,
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean detection records from one source dataset folder."""
    source_root = Path(source_root)
    if not source_root.exists():
        raise FileNotFoundError(f"Source dataset not found: {source_root.resolve()}")

    image_root = next(
        (p for p in [source_root / "JPEGImages", source_root / "images", source_root] if p.exists()),
        None,
    )
    if image_root is None:
        raise FileNotFoundError(f"No image root found below: {source_root.resolve()}")

    image_paths = sorted(
        p for p in image_root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found below: {image_root.resolve()}")

    image_names = [p.name for p in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError(
            f"Duplicate image filenames found below {source_root}; "
            "flat YOLO split folders require unique filenames within each dataset."
        )

    xml_by_stem = {p.stem: p for p in source_root.rglob("*.xml")}
    json_by_stem = {p.stem: p for p in source_root.rglob("*.json")}

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)
        annotation_size = None
        objects: List[Dict[str, Any]] = []
        source_used = None

        try:
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(xml_path)
                if xml_objects:
                    objects = xml_objects
                    source_used = "xml"

            # Fallback to JSON when XML is absent or produced no usable objects.
            if not objects and json_path is not None:
                objects = parse_json_annotation(json_path)
                if objects:
                    source_used = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []
        for obj in objects:
            label = normalize_label(obj["label"])
            if label not in TARGET_CLASS_SET:
                stats["non_target_objects_removed"] += 1
                continue

            clipped = clip_box(obj["bbox_xyxy"], width, height)
            if clipped is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({"label": label, "bbox_xyxy": clipped})

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        stats[f"used_{source_used}"] += 1
        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)

    if not records:
        raise RuntimeError(f"No usable records remained after filtering: {source_root}")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count images, boxes, and class occurrence statistics."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]
        for label in labels:
            box_counts[label] += 1
        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [name for name in class_names if box_counts[name] == 0],
    }


def plot_source_target_box_counts(
    source_audit: Dict[str, Any],
    target_audit: Dict[str, Any],
    output_root: Path,
) -> None:
    """Plot one grouped comparison of full-dataset box counts by class."""
    output_root.mkdir(parents=True, exist_ok=True)
    x = np.arange(len(CLASS_NAMES))
    width = 0.38

    plt.figure(figsize=(11, 5))
    plt.bar(
        x - width / 2,
        [source_audit["box_count_per_class"][name] for name in CLASS_NAMES],
        width,
        label=f"{TRAIN_DATASET_TAG} source",
    )
    plt.bar(
        x + width / 2,
        [target_audit["box_count_per_class"][name] for name in CLASS_NAMES],
        width,
        label=f"{TEST_DATASET_TAG} target",
    )
    plt.xticks(x, CLASS_NAMES, rotation=45, ha="right")
    plt.ylabel("Number of annotated boxes")
    plt.title(f"{CROSS_YEAR_TAG}: full-dataset class counts")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_root / "source_target_box_counts.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()


## Module 4: Deterministic cross-year splitting and Year2021 source subsampling

Splits the source and target years independently using 70/15/15. The effective layout is source train for training, source validation for validation, and target test for final testing. When Year2021 is the source year, only its training split is reduced to the target-year training-split size using the established primary-label stratification rule.


In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible 70/15/15 split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    shuffled = list(all_records)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    n_train = int(total * train_ratio)
    n_valid = int(total * valid_ratio)

    return {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train:n_train + n_valid],
        "test": shuffled[n_train + n_valid:],
    }


def get_record_primary_label(record: Dict[str, Any]) -> str:
    """
    Assign one primary class to an image for stratified subsampling.

    For multi-class images, use the most frequent class.
    Ties follow the fixed CLASS_NAMES order.
    """
    labels = [obj["label"] for obj in record.get("objects", [])]
    if not labels:
        return "unknown"

    counts = Counter(labels)
    return max(
        CLASS_NAMES,
        key=lambda class_name: (
            counts.get(class_name, 0),
            -CLASS_NAMES.index(class_name),
        ),
    )


def _exact_stratified_allocation(
    groups: Dict[str, List[Dict[str, Any]]],
    target_size: int,
) -> Dict[str, int]:
    """
    Allocate an exact target size approximately in proportion to group sizes,
    while retaining at least one record per represented group when possible.
    """
    labels = [label for label, group in groups.items() if group]
    total = sum(len(groups[label]) for label in labels)

    if target_size < 0 or target_size > total:
        raise ValueError("target_size must be between 0 and the number of available records.")
    if target_size == 0:
        return {label: 0 for label in labels}

    raw = {label: target_size * len(groups[label]) / total for label in labels}
    allocation = {label: min(int(math.floor(raw[label])), len(groups[label])) for label in labels}

    # Preserve represented classes where the requested sample is large enough.
    if target_size >= len(labels):
        for label in labels:
            if allocation[label] == 0:
                allocation[label] = 1

    # If minimum-one adjustments overshoot, remove from the most over-allocated
    # groups while keeping at least one when that rule is active.
    min_allowed = 1 if target_size >= len(labels) else 0
    while sum(allocation.values()) > target_size:
        reducible = [
            label for label in labels
            if allocation[label] > min_allowed
        ]
        if not reducible:
            raise RuntimeError("Could not reduce stratified allocation to the exact target size.")
        label = max(
            reducible,
            key=lambda name: (allocation[name] - raw[name], allocation[name], len(groups[name])),
        )
        allocation[label] -= 1

    # Fill remaining slots using largest remainder, respecting group capacity.
    while sum(allocation.values()) < target_size:
        candidates = [
            label for label in labels
            if allocation[label] < len(groups[label])
        ]
        if not candidates:
            raise RuntimeError("Could not fill stratified allocation to the exact target size.")
        label = max(
            candidates,
            key=lambda name: (raw[name] - allocation[name], len(groups[name])),
        )
        allocation[label] += 1

    if sum(allocation.values()) != target_size:
        raise RuntimeError("Stratified allocation did not reach the requested target size.")

    return allocation


def stratified_subsample_records(
    records: List[Dict[str, Any]],
    target_size: int,
    seed: int,
) -> List[Dict[str, Any]]:
    """Stratified image-level subsampling using the primary class of each image."""
    if target_size >= len(records):
        return list(records)

    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        groups[get_record_primary_label(record)].append(record)

    for group in groups.values():
        rng.shuffle(group)

    allocation = _exact_stratified_allocation(groups, target_size)

    sampled: List[Dict[str, Any]] = []
    for label, group in groups.items():
        sampled.extend(group[:allocation.get(label, 0)])

    rng.shuffle(sampled)

    if len(sampled) != target_size:
        raise RuntimeError(
            f"Subsampling produced {len(sampled)} records; expected exactly {target_size}."
        )
    return sampled


def maybe_subsample_source_train(
    source_train_records: List[Dict[str, Any]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    seed: int,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Apply the established subsampling rule only when Year2021 is the source year."""
    should_subsample = (
        SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE
        and TRAIN_YEAR == "2021"
    )

    info = {
        "enabled": bool(should_subsample),
        "source_year": TRAIN_YEAR,
        "target_year": TEST_YEAR,
        "mode": SUBSAMPLE_TARGET_MODE,
        "subsample_seed": None,
        "original_source_train_count": len(source_train_records),
        "target_size": None,
        "final_source_train_count": len(source_train_records),
    }

    if not should_subsample:
        return list(source_train_records), info

    if SUBSAMPLE_TARGET_MODE != "target_train_size":
        raise ValueError("This notebook freezes SUBSAMPLE_TARGET_MODE='target_train_size'.")

    target_size = len(target_splits["train"])
    subsample_seed = int(seed + SUBSAMPLE_SEED_OFFSET)

    if target_size > len(source_train_records):
        # Do not upsample. This guard makes the rule explicit and auditable.
        print(
            "WARNING: target train size exceeds Year2021 source train size; "
            "subsampling is skipped because the protocol does not upsample."
        )
        info["target_size"] = int(target_size)
        info["subsample_seed"] = subsample_seed
        return list(source_train_records), info

    sampled = stratified_subsample_records(
        records=source_train_records,
        target_size=target_size,
        seed=subsample_seed,
    )

    info["target_size"] = int(target_size)
    info["subsample_seed"] = subsample_seed
    info["final_source_train_count"] = len(sampled)
    return sampled, info


def _path_set(records: List[Dict[str, Any]]) -> set:
    return {str(Path(record["img_path"]).resolve()) for record in records}


def verify_cross_split_disjointness(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Fail if any physical image path occurs in more than one effective split."""
    train_paths = _path_set(cross_year_splits["train"])
    valid_paths = _path_set(cross_year_splits["valid"])
    test_paths = _path_set(cross_year_splits["test"])

    overlaps = {
        "train_valid": train_paths & valid_paths,
        "train_test": train_paths & test_paths,
        "valid_test": valid_paths & test_paths,
    }
    bad = {name: paths for name, paths in overlaps.items() if paths}
    if bad:
        raise RuntimeError(
            "Cross-year split leakage detected: "
            + ", ".join(f"{name}={len(paths)}" for name, paths in bad.items())
        )


def audit_cross_year_splits(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit effective train/valid/test class coverage."""
    report: Dict[str, Any] = {}
    for split_name, rows in cross_year_splits.items():
        split_audit = audit_records(rows, CLASS_NAMES)
        report[split_name] = split_audit
        print(
            f"{split_name:8s} images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )
        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Effective split '{split_name}' is missing classes: "
                f"{split_audit['missing_classes']}"
            )
    return report


def make_cross_year_splits_for_seed(
    seed: int,
    source_records: List[Dict[str, Any]],
    target_records: List[Dict[str, Any]],
) -> Tuple[
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    """
    Build source splits, target splits, and the effective cross-year layout.

    Effective YOLO layout:
        train = source train (subsampled only if source is Year2021)
        valid = source validation
        test  = target test
    """
    source_splits = split_records_deterministically(
        source_records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )
    target_splits = split_records_deterministically(
        target_records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )

    effective_source_train, subsample_info = maybe_subsample_source_train(
        source_train_records=source_splits["train"],
        target_splits=target_splits,
        seed=seed,
    )

    cross_year_splits = {
        "train": effective_source_train,
        "valid": source_splits["valid"],
        "test": target_splits["test"],
    }

    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    split_info = {
        "seed": int(seed),
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "source_dataset": TRAIN_DATASET_TAG,
        "target_dataset": TEST_DATASET_TAG,
        "ratios": {
            "train": TRAIN_RATIO,
            "valid": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "source_counts_original": {
            name: len(rows) for name, rows in source_splits.items()
        },
        "target_counts_original": {
            name: len(rows) for name, rows in target_splits.items()
        },
        "effective_cross_year_counts": {
            name: len(rows) for name, rows in cross_year_splits.items()
        },
        "subsample_info": subsample_info,
        "class_coverage": coverage,
    }

    return cross_year_splits, source_splits, target_splits, split_info


## Module 5: Export and verify the effective YOLO dataset

Writes the effective cross-year layout in YOLO format, saves filename-only split manifests, clears stale temporary split files, and verifies image and label counts before any training begins.


In [ ]:
def remove_file_or_link(path: Path) -> None:
    """Remove a regular file or symlink, including broken symlinks."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Clear files, symlinks, and old disk-cache .npy files from one split folder."""
    directory.mkdir(parents=True, exist_ok=True)
    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()


def place_image(source: Path, destination: Path, mode: str) -> str:
    """Place one image using symlink -> hardlink -> copy fallback."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError("mode must be 'symlink', 'hardlink', or 'copy'.")

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box
    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )
    return tuple(max(0.0, min(1.0, float(value))) for value in values)


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO label file."""
    lines: List[str] = []
    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]
        xc, yc, bw, bh = xyxy_to_yolo(
            obj["bbox_xyxy"], record["width"], record["height"]
        )
        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write an Ultralytics dataset.yaml for the effective cross-year layout."""
    yaml_path = dataset_root / "dataset.yaml"
    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/valid",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]
    lines.extend(f"  {idx}: {name}" for idx, name in enumerate(class_names))
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return yaml_path


def save_split_manifest(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_json: Path,
    seed: int,
    dataset_tag: str,
    role: str,
    extra_info: Optional[Dict[str, Any]] = None,
) -> None:
    """Save filenames and counts for reproducibility."""
    output_json.parent.mkdir(parents=True, exist_ok=True)
    manifest = {
        "dataset_tag": dataset_tag,
        "role": role,
        "seed": int(seed),
        "counts": {name: len(rows) for name, rows in split_records.items()},
        "files": {
            name: [record["file_name"] for record in rows]
            for name, rows in split_records.items()
        },
    }
    if extra_info is not None:
        manifest["extra_info"] = extra_info
    output_json.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def count_image_files(directory: Path) -> int:
    """Count only actual image files/symlinks, excluding disk-cache .npy files."""
    return sum(
        1 for path in directory.iterdir()
        if (path.is_file() or path.is_symlink())
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def count_label_files(directory: Path) -> int:
    return sum(
        1 for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".txt"
    )


def check_yolo_dataset_integrity(
    dataset_root: Path,
    expected_counts: Dict[str, int],
) -> None:
    """Verify expected image and label counts for every effective split."""
    for split_name in ("train", "valid", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name
        image_count = count_image_files(image_dir)
        label_count = count_label_files(label_dir)
        expected = int(expected_counts[split_name])

        print(
            f"{split_name:8s} images={image_count:5d} "
            f"labels={label_count:5d} expected={expected:5d}"
        )
        if image_count != expected or label_count != expected:
            raise RuntimeError(
                f"YOLO integrity failure in '{split_name}': "
                f"images={image_count}, labels={label_count}, expected={expected}."
            )


def prepare_cross_year_yolo_dataset(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    source_splits: Dict[str, List[Dict[str, Any]]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    split_info: Dict[str, Any],
    output_root: Path,
    manifest_root: Path,
    image_mode: str,
    seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Build and verify one effective cross-year YOLO dataset."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)
    class_to_id = {name: idx for idx, name in enumerate(CLASS_NAMES)}

    # Recheck leakage and class coverage immediately before export.
    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    image_dirs = {
        name: output_root / "images" / name for name in ("train", "valid", "test")
    }
    label_dirs = {
        name: output_root / "labels" / name for name in ("train", "valid", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {"symlinked": 0, "hardlinked": 0, "copied": 0}

    for split_name, rows in cross_year_splits.items():
        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]
            action = place_image(source, destination, image_mode)
            placement_counts[action] += 1

            label_path = label_dirs[split_name] / f"{Path(record['file_name']).stem}.txt"
            write_yolo_label(record, label_path, class_to_id)

    yaml_path = write_dataset_yaml(output_root, CLASS_NAMES)

    save_split_manifest(
        source_splits,
        manifest_root / f"{TRAIN_DATASET_TAG}_source_seed{seed}.json",
        seed,
        TRAIN_DATASET_TAG,
        "source",
    )
    save_split_manifest(
        target_splits,
        manifest_root / f"{TEST_DATASET_TAG}_target_seed{seed}.json",
        seed,
        TEST_DATASET_TAG,
        "target",
    )
    save_split_manifest(
        cross_year_splits,
        manifest_root / f"cross_{CROSS_YEAR_TAG}_seed{seed}.json",
        seed,
        CROSS_YEAR_TAG,
        "effective_cross_year",
        extra_info=split_info,
    )

    expected_counts = {
        name: len(rows) for name, rows in cross_year_splits.items()
    }
    check_yolo_dataset_integrity(output_root, expected_counts)

    layout_manifest = {
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "seed": int(seed),
        "dataset_yaml": portable_path(yaml_path),
        "output_root": portable_path(output_root),
        "placement_counts": placement_counts,
        "expected_counts": expected_counts,
        "split_info": split_info,
    }
    manifest_root.mkdir(parents=True, exist_ok=True)
    (manifest_root / "yolo_layout_manifest.json").write_text(
        json.dumps(layout_manifest, indent=2), encoding="utf-8"
    )

    print("Dataset YAML:", portable_path(yaml_path))
    print("Image placement:", placement_counts)
    return yaml_path, coverage

## Module 6: Source-validation objective and YOLOv8m training helper

Uses source-year validation mAP50-95 as the HPO objective. Every candidate starts from `yolov8m.pt`, uses the frozen fidelity-specific schedule, checks checkpoint alignment, and re-evaluates the saved best checkpoint on the source validation split.


In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract detection metrics; optimization fitness is mAP50-95 only."""
    box = getattr(metrics, "box", None)
    if box is None:
        raise RuntimeError("Ultralytics returned no detection box metrics.")

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_obj = getattr(metrics, "names", {})
    if isinstance(names_obj, dict):
        class_name_lookup = {int(k): str(v) for k, v in names_obj.items()}
    else:
        class_name_lookup = {idx: str(name) for idx, name in enumerate(names_obj)}

    per_class_rows: List[Dict[str, Any]] = []

    if hasattr(box, "ap") and hasattr(box, "ap_class_index"):
        ap_values = np.asarray(box.ap, dtype=float).reshape(-1)
        class_ids = np.asarray(box.ap_class_index, dtype=int).reshape(-1)

        for class_id, ap_value in zip(class_ids.tolist(), ap_values.tolist()):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })
    else:
        maps = [float(value) for value in box.maps]
        for class_id, ap_value in enumerate(maps):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })

    represented_ids = {int(row["class_id"]) for row in per_class_rows}
    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)

    if missing_ids:
        missing_names = [expected_class_names[idx] for idx in missing_ids]
        raise RuntimeError(
            "Validation/test metrics are missing represented classes: "
            f"{missing_names}. Check the split coverage."
        )

    sorted_aps = sorted(float(row["map50_95"]) for row in per_class_rows)
    bottom3 = float(mean(sorted_aps[:3])) if len(sorted_aps) >= 3 else float(mean(sorted_aps))

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "fitness": map50_95,
        "bottom3_mean_map50_95": bottom3,
        "per_class_map50_95": per_class_rows,
    }


def read_epoch_map50_95_values(results_csv: Path) -> Tuple[str, List[float]]:
    """Read epoch-level mAP50-95 values from an Ultralytics results.csv file."""
    results_csv = Path(results_csv)
    if not results_csv.exists():
        raise FileNotFoundError(f"Training results.csv not found: {results_csv}")

    with results_csv.open("r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [name for name in fieldnames if "mAP50-95" in name]
    if not candidate_columns:
        raise RuntimeError(
            f"Could not find an mAP50-95 column in {results_csv}. "
            f"Available columns: {fieldnames}"
        )

    metric_column = next(
        (name for name in candidate_columns if "metrics/" in name),
        candidate_columns[0],
    )

    values: List[float] = []
    for row in rows:
        raw_value = row.get(metric_column, "")
        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(f"No numeric values found in results column: {metric_column}")

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Verify that trainer.best_fitness agrees with maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(results_csv)
    max_epoch_map95 = float(max(map95_values))

    if trainer_best_fitness is None or not math.isfinite(float(trainer_best_fitness)):
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }
        if strict:
            raise RuntimeError(
                "Could not read Ultralytics trainer.best_fitness, so checkpoint-fitness "
                "alignment could not be verified."
            )
        return report

    trainer_best_fitness = float(trainer_best_fitness)
    difference = abs(trainer_best_fitness - max_epoch_map95)
    status = "PASS" if difference <= tolerance else "FAIL"

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Ultralytics checkpoint fitness does not align with mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max epoch mAP50-95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}."
        )

    return report


def train_and_validate_configuration(
    *,
    learning_rate: float,
    lrf: float,
    momentum: float,
    weight_decay: float,
    scale: float,
    dataset_yaml: Path,
    project_dir: Path,
    model_name: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    patience: int,
    workers: int,
    seed: int,
    warmup_epochs: float,
    close_mosaic: int,
    fixed_training_args: Dict[str, Any],
    expected_class_names: List[str],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Train from scratch, verify checkpoint alignment, then validate best.pt."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
    if learning_rate <= 0 or lrf <= 0:
        raise ValueError("learning_rate and lrf must be greater than 0.")
    if not 0.0 < momentum < 1.0:
        raise ValueError("momentum must be between 0 and 1.")
    if weight_decay < 0:
        raise ValueError("weight_decay cannot be negative.")
    if not 0.0 <= scale <= 1.0:
        raise ValueError("scale must be between 0 and 1.")
    if epochs < 1 or batch_size < 1 or workers < 0:
        raise ValueError("Invalid epochs, batch_size, or workers value.")
    if warmup_epochs < 0 or close_mosaic < 0:
        raise ValueError("warmup_epochs and close_mosaic must be non-negative.")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"
    using_cuda = str(device) != "cpu" and torch.cuda.is_available()

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    trial_id = uuid.uuid4().hex[:8]
    run_name = f"CY{TRAIN_YEAR}{TEST_YEAR}_s{seed}_e{epochs}_{trial_id}"
    project_dir.mkdir(parents=True, exist_ok=True)

    hyperparameters = {
        "learning_rate": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "epochs": int(epochs),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
        "seed": int(seed),
        "patience": int(patience),
    }

    print("\n" + "=" * 88)
    print("START TRAINING:", run_name)
    print(json.dumps(hyperparameters, indent=2))
    print("=" * 88)

    train_kwargs = dict(fixed_training_args)
    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "patience": int(patience),
        "workers": int(workers),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
    })

    model = YOLO(model_name)
    train_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = time.perf_counter() - train_start

    train_save_dir = getattr(train_result, "save_dir", None)
    if train_save_dir is None and getattr(model, "trainer", None) is not None:
        train_save_dir = getattr(model.trainer, "save_dir", None)
    if train_save_dir is None:
        raise RuntimeError("Training finished but save_dir could not be resolved.")

    train_save_dir = Path(train_save_dir)
    best_checkpoint = train_save_dir / "weights" / "best.pt"
    results_csv = train_save_dir / "results.csv"

    if not best_checkpoint.exists():
        raise FileNotFoundError(f"best.pt not found: {best_checkpoint}")
    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")

    trainer_best_fitness = None
    trainer = getattr(model, "trainer", None)
    if trainer is not None:
        raw_best_fitness = getattr(trainer, "best_fitness", None)
        if raw_best_fitness is not None:
            trainer_best_fitness = float(raw_best_fitness)

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=fitness_alignment_tolerance,
        strict=strict_fitness_alignment_check,
    )

    # Re-evaluate the frozen best checkpoint on the SOURCE validation split.
    best_model = YOLO(str(best_checkpoint))
    val_start = time.perf_counter()
    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_val",
        plots=False,
        save_json=False,
        verbose=False,
    )
    validation_seconds = time.perf_counter() - val_start

    metric_data = extract_box_metrics(validation_metrics, expected_class_names)

    peak_gpu_memory_gb = None
    if using_cuda:
        peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    result = {
        "status": "PASS",
        "trial_id": trial_id,
        "run_name": run_name,
        "cross_year_tag": CROSS_YEAR_TAG,
        "hyperparameters": hyperparameters,
        "metrics": metric_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(training_seconds),
            "validation_seconds": float(validation_seconds),
            "total_seconds": float(training_seconds + validation_seconds),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0) if using_cuda else None,
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb) if peak_gpu_memory_gb is not None else None
            ),
        },
        "paths": {
            "training_folder": portable_path(train_save_dir),
            "best_checkpoint": portable_path(best_checkpoint),
            "results_csv": portable_path(results_csv),
        },
    }

    (train_save_dir / "trial_summary.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )

    print(
        f"Completed {run_name}: source-val mAP50-95={metric_data['map50_95']:.6f}, "
        f"P={metric_data['precision']:.6f}, R={metric_data['recall']:.6f}, "
        f"alignment={alignment_report['status']}"
    )
    return result

## Module 7: AF-MCSO search geometry and adaptive-fidelity prediction

Represents the five tuned hyperparameters in normalized search space. Initial candidates use maximin diversity, later candidates combine Levy-based local search with opposition and global exploration, and completed 20-to-40 epoch validation pairs guide promotion. Target-year test data is never used here.


In [ ]:
def clip01(value: float) -> float:
    return max(0.0, min(1.0, float(value)))


def reflect01(value: float) -> float:
    """Reflect a coordinate into [0,1] without clipping away overshoot information."""
    value = float(value)
    while value < 0.0 or value > 1.0:
        if value < 0.0:
            value = -value
        if value > 1.0:
            value = 2.0 - value
    return clip01(value)


def vector_signature(vector: List[float]) -> Tuple[float, ...]:
    return tuple(round(float(value), SIGNATURE_PRECISION) for value in vector)


def normalized_distance(a: List[float], b: List[float]) -> float:
    return math.sqrt(sum((float(x) - float(y)) ** 2 for x, y in zip(a, b)))


def mean_pairwise_distance(vectors: List[List[float]]) -> float:
    if len(vectors) < 2:
        return 0.0
    distances = []
    for i in range(len(vectors)):
        for j in range(i + 1, len(vectors)):
            distances.append(normalized_distance(vectors[i], vectors[j]))
    return float(mean(distances)) if distances else 0.0


def decode_vector(vector: List[float], config_id: Optional[str] = None) -> Dict[str, Any]:
    """Decode normalized search coordinates into the frozen 5D parameter space."""
    if len(vector) != len(SEARCH_PARAMETER_ORDER):
        raise ValueError("Vector dimensionality does not match SEARCH_PARAMETER_ORDER.")

    config: Dict[str, Any] = {}
    for parameter_name, z in zip(SEARCH_PARAMETER_ORDER, vector):
        z = clip01(z)
        spec = SEARCH_SPACE[parameter_name]
        low = float(spec["low"])
        high = float(spec["high"])

        if spec["distribution"] == "log":
            value = math.exp(math.log(low) + z * (math.log(high) - math.log(low)))
        elif spec["distribution"] == "linear":
            value = low + z * (high - low)
        else:
            raise ValueError(f"Unsupported distribution for {parameter_name}: {spec['distribution']}")

        config[parameter_name] = float(value)

    if config_id is not None:
        config["config_id"] = str(config_id)
    return config


def encode_config(config: Dict[str, Any]) -> List[float]:
    """Encode a 5D configuration into normalized coordinates."""
    vector = []
    for parameter_name in SEARCH_PARAMETER_ORDER:
        spec = SEARCH_SPACE[parameter_name]
        low = float(spec["low"])
        high = float(spec["high"])
        value = float(config[parameter_name])

        if spec["distribution"] == "log":
            z = (math.log(value) - math.log(low)) / (math.log(high) - math.log(low))
        else:
            z = (value - low) / (high - low)
        vector.append(clip01(z))
    return vector


def row_to_vector(row: Dict[str, Any]) -> List[float]:
    return [float(row[f"z{index}"]) for index in range(len(SEARCH_PARAMETER_ORDER))]


def sample_unit_vector(rng: random.Random) -> List[float]:
    return [rng.random() for _ in SEARCH_PARAMETER_ORDER]


def maximin_initial_vectors(
    rng: random.Random,
    n_vectors: int,
    pool_size: int,
) -> List[List[float]]:
    """Choose a diverse initial population from a seeded random candidate pool."""
    if n_vectors < 2:
        raise ValueError("n_vectors must be at least 2.")
    if pool_size < n_vectors:
        raise ValueError("pool_size must be at least n_vectors.")

    pool = [sample_unit_vector(rng) for _ in range(pool_size)]

    # Start with the farthest pair.
    best_pair = None
    best_distance = -1.0
    for i in range(len(pool)):
        for j in range(i + 1, len(pool)):
            distance = normalized_distance(pool[i], pool[j])
            if distance > best_distance:
                best_distance = distance
                best_pair = (i, j)

    if best_pair is None:
        raise RuntimeError("Could not create a maximin initial pair.")

    selected = [pool[best_pair[0]], pool[best_pair[1]]]
    selected_signatures = {vector_signature(vector) for vector in selected}

    while len(selected) < n_vectors:
        best_candidate = None
        best_min_distance = -1.0
        for candidate in pool:
            if vector_signature(candidate) in selected_signatures:
                continue
            min_distance = min(normalized_distance(candidate, existing) for existing in selected)
            if min_distance > best_min_distance:
                best_min_distance = min_distance
                best_candidate = candidate

        if best_candidate is None:
            raise RuntimeError("Maximin initialization could not find enough unique vectors.")

        selected.append(best_candidate)
        selected_signatures.add(vector_signature(best_candidate))

    return selected


def levy_step(rng: random.Random, beta: float = LEVY_BETA) -> float:
    """Mantegna Levy-flight step."""
    sigma_u = (
        (
            math.gamma(1.0 + beta)
            * math.sin(math.pi * beta / 2.0)
        )
        /
        (
            math.gamma((1.0 + beta) / 2.0)
            * beta
            * (2.0 ** ((beta - 1.0) / 2.0))
        )
    ) ** (1.0 / beta)

    u = rng.gauss(0.0, sigma_u)
    v = rng.gauss(0.0, 1.0)
    denominator = max(abs(v) ** (1.0 / beta), 1e-12)
    return max(-5.0, min(5.0, u / denominator))


def local_levy_child(
    parent: List[float],
    rng: random.Random,
    step_scale: float,
) -> List[float]:
    """Exploit around the current full-fidelity elite."""
    child = list(parent)
    coordinate_count = rng.randint(1, min(2, len(child)))
    indices = rng.sample(range(len(child)), k=coordinate_count)
    for index in indices:
        child[index] = reflect01(child[index] + step_scale * levy_step(rng))
    return child


def opposition_levy_child(
    parent: List[float],
    rng: random.Random,
    step_scale: float,
    direction_weight: float,
) -> List[float]:
    """Explore toward the opposite region with a small Levy perturbation."""
    child = list(parent)
    coordinate_count = rng.randint(2, min(4, len(child)))
    indices = rng.sample(range(len(child)), k=coordinate_count)

    for index in indices:
        opposite = 1.0 - parent[index]
        directed = parent[index] + direction_weight * (opposite - parent[index])
        child[index] = reflect01(directed + 0.5 * step_scale * levy_step(rng))
    return child


def maximin_immigrant(
    existing_vectors: List[List[float]],
    rng: random.Random,
    pool_size: int = IMMIGRANT_POOL_SIZE,
) -> List[float]:
    """Choose a global candidate far from the already evaluated region."""
    pool = [sample_unit_vector(rng) for _ in range(pool_size)]
    if not existing_vectors:
        return pool[0]

    return max(
        pool,
        key=lambda candidate: min(
            normalized_distance(candidate, existing)
            for existing in existing_vectors
        ),
    )


def ensure_unique_vector(
    proposed_vector: List[float],
    registry: set,
    existing_vectors: List[List[float]],
    rng: random.Random,
) -> List[float]:
    if vector_signature(proposed_vector) not in registry:
        return proposed_vector

    for _ in range(50):
        replacement = maximin_immigrant(
            existing_vectors=existing_vectors,
            rng=rng,
            pool_size=100,
        )
        if vector_signature(replacement) not in registry:
            return replacement

    raise RuntimeError("Could not generate a unique AF-MCSO candidate.")


def mcso_rank_key(row: Dict[str, Any]) -> Tuple[float, float, float, str]:
    """Primary objective is validation mAP50-95; other metrics only break ties."""
    return (
        -float(row["val_map50_95"]),
        -float(row["val_map50"]),
        -float(row["val_recall"]),
        str(row["candidate_id"]),
    )


def rank_mcso_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    successful = [
        row for row in rows
        if row.get("status") == "PASS"
        and row.get("val_map50_95", "") != ""
    ]
    return sorted(successful, key=mcso_rank_key)


def build_paired_fidelity_history(
    all_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Build successful 20->40 validation pairs for the same configuration."""
    grouped: Dict[str, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    for row in all_rows:
        if row.get("status") != "PASS" or row.get("val_map50_95", "") == "":
            continue
        budget = int(row["budget_epochs"])
        if budget in {LOW_FIDELITY_EPOCHS, FULL_FIDELITY_EPOCHS}:
            grouped[str(row["candidate_id"])][budget] = row

    history = []
    for candidate_id, by_budget in grouped.items():
        if LOW_FIDELITY_EPOCHS not in by_budget or FULL_FIDELITY_EPOCHS not in by_budget:
            continue

        low_row = by_budget[LOW_FIDELITY_EPOCHS]
        full_row = by_budget[FULL_FIDELITY_EPOCHS]
        map20 = float(low_row["val_map50_95"])
        map40 = float(full_row["val_map50_95"])

        history.append({
            "candidate_id": candidate_id,
            "vector": row_to_vector(low_row),
            "map50_95_20": map20,
            "map50_95_40": map40,
            "gain_20_to_40": map40 - map20,
        })

    return sorted(history, key=lambda item: item["candidate_id"])


def predict_full_fidelity_map50_95(
    low_row: Dict[str, Any],
    paired_history: List[Dict[str, Any]],
) -> Dict[str, float]:
    """
    Predict 40-epoch mAP50-95 using inverse-distance weighted gains from
    completed nearby 20->40 pairs. This affects promotion only.
    """
    observed20 = float(low_row["val_map50_95"])

    if not paired_history:
        return {
            "predicted_gain": 0.0,
            "predicted_map50_95_40": observed20,
            "history_pairs_used": 0,
        }

    target_vector = row_to_vector(low_row)
    distance_rows = [
        (normalized_distance(target_vector, item["vector"]), item)
        for item in paired_history
    ]
    distance_rows.sort(key=lambda pair: pair[0])

    k = min(AF_K_NEIGHBORS, len(distance_rows))
    nearest = distance_rows[:k]

    if nearest[0][0] <= AF_DISTANCE_EPS:
        predicted_gain = float(nearest[0][1]["gain_20_to_40"])
    else:
        weighted_gain = 0.0
        weight_sum = 0.0
        for distance, item in nearest:
            weight = 1.0 / max(distance, AF_DISTANCE_EPS)
            weighted_gain += weight * float(item["gain_20_to_40"])
            weight_sum += weight
        predicted_gain = weighted_gain / weight_sum if weight_sum > 0 else 0.0

    predicted40 = clip01(observed20 + predicted_gain)
    return {
        "predicted_gain": float(predicted_gain),
        "predicted_map50_95_40": float(predicted40),
        "history_pairs_used": int(k),
    }


def score_low_fidelity_candidates(
    low_rows: List[Dict[str, Any]],
    paired_history: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Attach predicted 40-epoch mAP50-95 and rank candidates for promotion."""
    successful = rank_mcso_rows(low_rows)

    if len(paired_history) < AF_MIN_HISTORY_PAIRS:
        raise RuntimeError(
            f"AF promotion requires at least {AF_MIN_HISTORY_PAIRS} completed "
            f"20->40 pairs; found {len(paired_history)}."
        )

    for row in successful:
        prediction = predict_full_fidelity_map50_95(
            low_row=row,
            paired_history=paired_history,
        )
        row["af_predicted_gain"] = prediction["predicted_gain"]
        row["af_predicted_map50_95_40"] = prediction["predicted_map50_95_40"]
        row["af_history_pairs_used"] = prediction["history_pairs_used"]

    return sorted(
        successful,
        key=lambda row: (
            -float(row["af_predicted_map50_95_40"]),
            -float(row["val_map50_95"]),
            -float(row["val_map50"]),
            -float(row["val_recall"]),
            str(row["candidate_id"]),
        ),
    )


def build_full_fidelity_archive(
    full_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Keep strong but reasonably distinct 40-epoch configurations as search parents."""
    ranked = rank_mcso_rows(full_rows)
    if not ranked:
        raise RuntimeError("Cannot build an AF-MCSO archive without 40-epoch results.")

    archive: List[Dict[str, Any]] = []
    for row in ranked:
        vector = row_to_vector(row)
        if not archive:
            archive.append(row)
        else:
            min_distance = min(
                normalized_distance(vector, row_to_vector(existing))
                for existing in archive
            )
            if min_distance >= ARCHIVE_MIN_DISTANCE:
                archive.append(row)

        if len(archive) >= ELITE_ARCHIVE_SIZE:
            break

    if len(archive) < min(ELITE_ARCHIVE_SIZE, len(ranked)):
        existing_ids = {row["eval_id"] for row in archive}
        for row in ranked:
            if row["eval_id"] in existing_ids:
                continue
            archive.append(row)
            existing_ids.add(row["eval_id"])
            if len(archive) >= min(ELITE_ARCHIVE_SIZE, len(ranked)):
                break

    return archive


def choose_initial_promotions(
    low_rows: List[Dict[str, Any]],
) -> List[Tuple[Dict[str, Any], str]]:
    """
    Bootstrap AF with two full-fidelity pairs:
    1) best 20-epoch mAP50-95;
    2) a diversity rescue farthest from the first among the remaining candidates.
    """
    ranked = rank_mcso_rows(low_rows)
    if len(ranked) < INITIAL_CONFIRMATIONS:
        raise RuntimeError("Not enough successful Stage-0 candidates for promotion.")

    first = ranked[0]
    remaining = ranked[1:]
    first_vector = row_to_vector(first)
    rescue = max(
        remaining,
        key=lambda row: normalized_distance(row_to_vector(row), first_vector),
    )

    return [
        (first, "initial_best_map50_95"),
        (rescue, "initial_diversity_rescue"),
    ]


def choose_generation_promotion(
    low_rows: List[Dict[str, Any]],
    paired_history: List[Dict[str, Any]],
) -> Tuple[Dict[str, Any], str]:
    ranked = score_low_fidelity_candidates(
        low_rows=low_rows,
        paired_history=paired_history,
    )
    if not ranked:
        raise RuntimeError("No successful generation candidate is available for promotion.")
    return ranked[0], "adaptive_fidelity_predicted_40ep_map50_95"


def generate_generation_proposals(
    *,
    archive_rows: List[Dict[str, Any]],
    evaluated_vectors: List[List[float]],
    registry: set,
    rng: random.Random,
    generation: int,
    previous_generation_improved: bool,
) -> Tuple[List[Tuple[str, List[float]]], str, float]:
    """
    Generate exactly two real low-fidelity candidates:
    - one local Levy exploitation candidate;
    - one exploration candidate selected, at zero training cost, between an
      opposition-Levy proposal and a maximin global immigrant.
    """
    if not archive_rows:
        raise RuntimeError("AF-MCSO needs a full-fidelity archive before evolution.")

    base_scale = GENERATION_STEP_SCALES[generation - 1]
    stagnated = generation > 1 and not previous_generation_improved

    if stagnated:
        step_scale = base_scale * STAGNATION_SCALE_MULTIPLIER
        opposition_weight = STAGNATION_OPPOSITION_WEIGHT
        mode = "stagnation_escape"
    else:
        step_scale = base_scale
        opposition_weight = OPPOSITION_DIRECTION_WEIGHT
        mode = "normal"

    best_parent = row_to_vector(archive_rows[0])
    diverse_parent = row_to_vector(archive_rows[1]) if len(archive_rows) > 1 else best_parent

    exploit = local_levy_child(
        parent=best_parent,
        rng=rng,
        step_scale=step_scale,
    )
    exploit = ensure_unique_vector(
        proposed_vector=exploit,
        registry=registry,
        existing_vectors=evaluated_vectors,
        rng=rng,
    )

    temp_existing = list(evaluated_vectors) + [exploit]

    opposition = opposition_levy_child(
        parent=diverse_parent,
        rng=rng,
        step_scale=step_scale,
        direction_weight=opposition_weight,
    )
    immigrant = maximin_immigrant(
        existing_vectors=temp_existing,
        rng=rng,
        pool_size=IMMIGRANT_POOL_SIZE,
    )

    def min_distance_to_existing(vector: List[float]) -> float:
        return min(normalized_distance(vector, existing) for existing in temp_existing)

    if min_distance_to_existing(opposition) >= min_distance_to_existing(immigrant):
        explore_role = "opposition_levy"
        explore = opposition
    else:
        explore_role = "maximin_immigrant"
        explore = immigrant

    explore = ensure_unique_vector(
        proposed_vector=explore,
        registry=registry | {vector_signature(exploit)},
        existing_vectors=temp_existing,
        rng=rng,
    )

    return [
        ("elite_levy", exploit),
        (explore_role, explore),
    ], mode, float(step_scale)


def save_rows_to_csv_safe(rows: List[Dict[str, Any]], output_csv: Path) -> None:
    if not rows:
        return
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: List[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    with output_csv.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_json(payload: Dict[str, Any], output_path: Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def summarize_numeric_rows(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    summary_rows = []
    for metric_name in metric_names:
        values = [float(row[metric_name]) for row in rows if row.get(metric_name, "") != ""]
        if not values:
            continue
        summary_rows.append({
            "metric": metric_name,
            "n": len(values),
            "mean": float(mean(values)),
            "sd": float(stdev(values)) if len(values) >= 2 else "",
            "min": float(min(values)),
            "max": float(max(values)),
        })
    return summary_rows


def summarize_per_class_rows(
    per_class_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    grouped: Dict[Tuple[int, str], List[float]] = defaultdict(list)
    for row in per_class_rows:
        grouped[(int(row["class_id"]), str(row["class_name"]))].append(
            float(row["target_test_map50_95"])
        )

    output = []
    for (class_id, class_name), values in sorted(grouped.items()):
        output.append({
            "class_id": class_id,
            "class_name": class_name,
            "n": len(values),
            "mean_target_test_map50_95": float(mean(values)),
            "sd_target_test_map50_95": float(stdev(values)) if len(values) >= 2 else "",
            "min_target_test_map50_95": float(min(values)),
            "max_target_test_map50_95": float(max(values)),
        })
    return output


def build_boundary_report(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []
    vector = encode_config(config)
    for parameter_name, z in zip(SEARCH_PARAMETER_ORDER, vector):
        spec = SEARCH_SPACE[parameter_name]
        rows.append({
            "parameter": parameter_name,
            "selected_value": float(config[parameter_name]),
            "low": float(spec["low"]),
            "high": float(spec["high"]),
            "distribution": spec["distribution"],
            "normalized_position": float(z),
            "distance_to_nearest_boundary": float(min(z, 1.0 - z)),
        })
    return rows


## Module 8: Exact 320 epoch-unit AF-MCSO engine

Runs Stage 0 with `4 x 20 + 2 x 40 = 160` epoch-units, followed by two generations of `2 x 20 + 1 x 40 = 80` epoch-units each. The HPO budget is therefore exactly 320 epoch-units. Only measured 40-epoch source-validation mAP50-95 can determine the final HPO winner.


In [ ]:
def reserve_hpo_budget(
    budget_tracker: Dict[str, int],
    budget_epochs: int,
) -> None:
    """Count a scheduled evaluation before training so failures cannot be silently replaced."""
    budget_epochs = int(budget_epochs)
    projected = budget_tracker["attempted_epoch_units"] + budget_epochs
    if projected > PLANNED_HPO_EPOCH_UNITS:
        raise RuntimeError(
            f"AF-MCSO would exceed the frozen {PLANNED_HPO_EPOCH_UNITS}-unit HPO budget."
        )

    budget_tracker["attempted_epoch_units"] = projected
    budget_tracker["attempted_evaluations"] += 1
    if budget_epochs == LOW_FIDELITY_EPOCHS:
        budget_tracker["attempted_20epoch"] += 1
    elif budget_epochs == FULL_FIDELITY_EPOCHS:
        budget_tracker["attempted_40epoch"] += 1
    else:
        raise ValueError(f"Unexpected AF-MCSO fidelity: {budget_epochs}")


def evaluate_mcso_candidate(
    *,
    candidate_id: str,
    vector: List[float],
    budget_epochs: int,
    stage_name: str,
    generation: int,
    role: str,
    promotion_reason: str,
    optimization_dataset_yaml: Path,
    device: str,
    budget_tracker: Dict[str, int],
) -> Dict[str, Any]:
    """Evaluate one AF-MCSO configuration at 20 or 40 epochs."""
    reserve_hpo_budget(budget_tracker, budget_epochs)
    evaluation_start = time.perf_counter()

    config = decode_vector(vector, config_id=candidate_id)
    warmup_epochs, close_mosaic = get_fidelity_controls(budget_epochs)

    try:
        result = train_and_validate_configuration(
            learning_rate=config["learning_rate"],
            lrf=config["lrf"],
            momentum=config["momentum"],
            weight_decay=config["weight_decay"],
            scale=config["scale"],
            dataset_yaml=optimization_dataset_yaml,
            project_dir=HPO_RUNS_ROOT,
            model_name=MODEL_NAME,
            epochs=int(budget_epochs),
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            patience=PATIENCE,
            workers=WORKERS,
            seed=OPTIMIZATION_TRAIN_SEED,
            warmup_epochs=float(warmup_epochs),
            close_mosaic=int(close_mosaic),
            fixed_training_args=FIXED_TRAINING_ARGS,
            expected_class_names=CLASS_NAMES,
            strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
            fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
            device=device,
        )

        metrics = result["metrics"]
        wall_seconds = time.perf_counter() - evaluation_start

        row = {
            "status": "PASS",
            "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR,
            "test_year": TEST_YEAR,
            "eval_id": f"{candidate_id}_E{int(budget_epochs)}",
            "candidate_id": candidate_id,
            "stage": stage_name,
            "generation": int(generation),
            "role": role,
            "promotion_reason": promotion_reason,
            "budget_epochs": int(budget_epochs),
            "learning_rate": float(config["learning_rate"]),
            "lrf": float(config["lrf"]),
            "momentum": float(config["momentum"]),
            "weight_decay": float(config["weight_decay"]),
            "scale": float(config["scale"]),
            "val_precision": float(metrics["precision"]),
            "val_recall": float(metrics["recall"]),
            "val_map50": float(metrics["map50"]),
            "val_map50_95": float(metrics["map50_95"]),
            "val_bottom3_mean_map50_95": float(metrics["bottom3_mean_map50_95"]),
            "training_seconds": float(result["timing"]["training_seconds"]),
            "validation_seconds": float(result["timing"]["validation_seconds"]),
            "evaluation_wall_seconds": float(wall_seconds),
            "af_predicted_gain": "",
            "af_predicted_map50_95_40": "",
            "af_history_pairs_used": "",
            "af_observed_gain": "",
            "best_checkpoint": result["paths"]["best_checkpoint"],
            "error_message": "",
        }
    except Exception as error:
        wall_seconds = time.perf_counter() - evaluation_start
        print("Candidate evaluation failed:", sanitize_exception_message(error))

        row = {
            "status": "FAIL",
            "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR,
            "test_year": TEST_YEAR,
            "eval_id": f"{candidate_id}_E{int(budget_epochs)}",
            "candidate_id": candidate_id,
            "stage": stage_name,
            "generation": int(generation),
            "role": role,
            "promotion_reason": promotion_reason,
            "budget_epochs": int(budget_epochs),
            "learning_rate": float(config["learning_rate"]),
            "lrf": float(config["lrf"]),
            "momentum": float(config["momentum"]),
            "weight_decay": float(config["weight_decay"]),
            "scale": float(config["scale"]),
            "val_precision": "",
            "val_recall": "",
            "val_map50": "",
            "val_map50_95": "",
            "val_bottom3_mean_map50_95": "",
            "training_seconds": "",
            "validation_seconds": "",
            "evaluation_wall_seconds": float(wall_seconds),
            "af_predicted_gain": "",
            "af_predicted_map50_95_40": "",
            "af_history_pairs_used": "",
            "af_observed_gain": "",
            "best_checkpoint": "",
            "error_message": sanitize_exception_message(error),
        }

    for index, value in enumerate(vector):
        row[f"z{index}"] = float(value)

    return row


def confirm_promoted_candidates(
    *,
    selected_pairs: List[Tuple[Dict[str, Any], str]],
    stage_name: str,
    generation: int,
    optimization_dataset_yaml: Path,
    device: str,
    all_rows: List[Dict[str, Any]],
    promotion_rows: List[Dict[str, Any]],
    results_csv: Path,
    promotions_csv: Path,
    budget_tracker: Dict[str, int],
) -> List[Dict[str, Any]]:
    """Retrain selected 20-epoch configurations independently at 40 epochs."""
    confirmed_rows = []

    for low_row, reason in selected_pairs:
        full_row = evaluate_mcso_candidate(
            candidate_id=str(low_row["candidate_id"]),
            vector=row_to_vector(low_row),
            budget_epochs=FULL_FIDELITY_EPOCHS,
            stage_name=stage_name,
            generation=generation,
            role=str(low_row["role"]),
            promotion_reason=reason,
            optimization_dataset_yaml=optimization_dataset_yaml,
            device=device,
            budget_tracker=budget_tracker,
        )

        full_row["af_predicted_gain"] = low_row.get("af_predicted_gain", "")
        full_row["af_predicted_map50_95_40"] = low_row.get(
            "af_predicted_map50_95_40", ""
        )
        full_row["af_history_pairs_used"] = low_row.get("af_history_pairs_used", "")

        if full_row["status"] == "PASS":
            observed_gain = (
                float(full_row["val_map50_95"]) - float(low_row["val_map50_95"])
            )
            low_row["af_observed_gain"] = observed_gain
            full_row["af_observed_gain"] = observed_gain

        all_rows.append(full_row)
        confirmed_rows.append(full_row)

        promotion_rows.append({
            "candidate_id": low_row["candidate_id"],
            "stage": stage_name,
            "generation": int(generation),
            "promotion_reason": reason,
            "map50_95_20": low_row.get("val_map50_95", ""),
            "predicted_gain_20_to_40": low_row.get("af_predicted_gain", ""),
            "predicted_map50_95_40": low_row.get("af_predicted_map50_95_40", ""),
            "history_pairs_used": low_row.get("af_history_pairs_used", ""),
            "map50_95_40": full_row.get("val_map50_95", ""),
            "observed_gain_20_to_40": full_row.get("af_observed_gain", ""),
            "status_40": full_row["status"],
        })

        save_rows_to_csv_safe(all_rows, results_csv)
        save_rows_to_csv_safe(promotion_rows, promotions_csv)

    return confirmed_rows


def require_all_pass(rows: List[Dict[str, Any]], context: str) -> None:
    failed = [row for row in rows if row.get("status") != "PASS"]
    if failed:
        failed_ids = [row["eval_id"] for row in failed]
        raise RuntimeError(
            f"{context} has failed scheduled evaluations: {failed_ids}. "
            "They are not silently replaced because the HPO budget is frozen."
        )


def run_af_mcso_hpo(
    *,
    optimization_dataset_yaml: Path,
    device: str,
    results_csv: Path,
    promotions_csv: Path,
    stage_summary_csv: Path,
) -> Tuple[
    Dict[str, Any],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
]:
    """Run the exact 320-unit AF-MCSO HPO schedule."""
    rng = random.Random(MCSO_RANDOM_SEED)

    all_rows: List[Dict[str, Any]] = []
    promotion_rows: List[Dict[str, Any]] = []
    stage_summary_rows: List[Dict[str, Any]] = []

    vector_registry = set()
    evaluated_vectors: List[List[float]] = []
    candidate_counter = 0

    budget_tracker = {
        "attempted_epoch_units": 0,
        "attempted_evaluations": 0,
        "attempted_20epoch": 0,
        "attempted_40epoch": 0,
    }

    # ========================================================
    # Stage 0: diverse 20-epoch initialization + two confirmations
    # ========================================================
    print("\n" + "=" * 100)
    print("AF-MCSO STAGE 0 : 4 MAXIMIN CANDIDATES @20 -> 2 CONFIRMATIONS @40")
    print("=" * 100)

    initial_vectors = maximin_initial_vectors(
        rng=rng,
        n_vectors=INITIAL_NESTS,
        pool_size=INITIAL_MAXIMIN_POOL,
    )

    initial_low_rows = []
    for vector in initial_vectors:
        candidate_counter += 1
        candidate_id = f"N{candidate_counter:03d}"

        signature = vector_signature(vector)
        if signature in vector_registry:
            raise RuntimeError("Duplicate Stage-0 vector.")
        vector_registry.add(signature)
        evaluated_vectors.append(vector)

        row = evaluate_mcso_candidate(
            candidate_id=candidate_id,
            vector=vector,
            budget_epochs=LOW_FIDELITY_EPOCHS,
            stage_name="stage0_initial",
            generation=0,
            role="maximin_initial",
            promotion_reason="",
            optimization_dataset_yaml=optimization_dataset_yaml,
            device=device,
            budget_tracker=budget_tracker,
        )
        all_rows.append(row)
        initial_low_rows.append(row)
        save_rows_to_csv_safe(all_rows, results_csv)

    require_all_pass(initial_low_rows, "Stage-0 low-fidelity initialization")

    initial_promotions = choose_initial_promotions(initial_low_rows)
    initial_full_rows = confirm_promoted_candidates(
        selected_pairs=initial_promotions,
        stage_name="stage0_initial",
        generation=0,
        optimization_dataset_yaml=optimization_dataset_yaml,
        device=device,
        all_rows=all_rows,
        promotion_rows=promotion_rows,
        results_csv=results_csv,
        promotions_csv=promotions_csv,
        budget_tracker=budget_tracker,
    )
    require_all_pass(initial_full_rows, "Stage-0 full-fidelity confirmations")

    paired_history = build_paired_fidelity_history(all_rows)
    if len(paired_history) != INITIAL_CONFIRMATIONS:
        raise RuntimeError(
            f"Stage 0 should create {INITIAL_CONFIRMATIONS} completed 20->40 pairs; "
            f"found {len(paired_history)}."
        )

    initial_diversity = mean_pairwise_distance(
        [row_to_vector(row) for row in initial_low_rows]
    )
    incumbent = float(rank_mcso_rows(initial_full_rows)[0]["val_map50_95"])

    stage_summary_rows.append({
        "stage": "stage0_initial",
        "generation": 0,
        "mode": "initialization",
        "candidate_count_20": len(initial_low_rows),
        "confirmation_count_40": len(initial_full_rows),
        "mean_pairwise_distance_20": initial_diversity,
        "best_map50_95_20": float(rank_mcso_rows(initial_low_rows)[0]["val_map50_95"]),
        "best_confirmed_map50_95_40": incumbent,
        "incumbent_map50_95_40": incumbent,
        "paired_fidelity_history_size": len(paired_history),
        "best_predicted_map50_95_40": "",
        "improved_full_fidelity": True,
    })
    save_rows_to_csv_safe(stage_summary_rows, stage_summary_csv)

    previous_generation_improved = True

    # ========================================================
    # Generations 1 and 2: two 20-epoch proposals -> one AF promotion
    # ========================================================
    for generation in range(1, MCSO_GENERATIONS + 1):
        print("\n" + "#" * 100)
        print(f"AF-MCSO GENERATION {generation}/{MCSO_GENERATIONS}")
        print("#" * 100)

        successful_full_rows = [
            row for row in all_rows
            if row["status"] == "PASS"
            and int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
        ]
        archive = build_full_fidelity_archive(successful_full_rows)
        incumbent_before = float(rank_mcso_rows(successful_full_rows)[0]["val_map50_95"])

        proposals, mode, step_scale = generate_generation_proposals(
            archive_rows=archive,
            evaluated_vectors=evaluated_vectors,
            registry=vector_registry,
            rng=rng,
            generation=generation,
            previous_generation_improved=previous_generation_improved,
        )

        generation_low_rows = []
        for role, vector in proposals:
            signature = vector_signature(vector)
            if signature in vector_registry:
                raise RuntimeError(f"Duplicate proposal produced in generation {generation}.")
            vector_registry.add(signature)
            evaluated_vectors.append(vector)

            candidate_counter += 1
            candidate_id = f"G{generation}C{candidate_counter:03d}"

            row = evaluate_mcso_candidate(
                candidate_id=candidate_id,
                vector=vector,
                budget_epochs=LOW_FIDELITY_EPOCHS,
                stage_name=f"generation_{generation}",
                generation=generation,
                role=role,
                promotion_reason="",
                optimization_dataset_yaml=optimization_dataset_yaml,
                device=device,
                budget_tracker=budget_tracker,
            )
            all_rows.append(row)
            generation_low_rows.append(row)
            save_rows_to_csv_safe(all_rows, results_csv)

        require_all_pass(generation_low_rows, f"Generation {generation} low-fidelity search")

        # Only earlier completed pairs are used to predict the candidates' 40-epoch performance.
        paired_history = build_paired_fidelity_history(all_rows)
        ranked_for_promotion = score_low_fidelity_candidates(
            low_rows=generation_low_rows,
            paired_history=paired_history,
        )

        print("\nAdaptive-fidelity promotion ranking:")
        for rank, row in enumerate(ranked_for_promotion, start=1):
            print(
                f"#{rank} {row['candidate_id']} | "
                f"mAP95@20={float(row['val_map50_95']):.6f} | "
                f"pred_gain={float(row['af_predicted_gain']):+.6f} | "
                f"pred_mAP95@40={float(row['af_predicted_map50_95_40']):.6f}"
            )

        promoted_row, reason = choose_generation_promotion(
            low_rows=generation_low_rows,
            paired_history=paired_history,
        )
        confirmed_rows = confirm_promoted_candidates(
            selected_pairs=[(promoted_row, reason)],
            stage_name=f"generation_{generation}",
            generation=generation,
            optimization_dataset_yaml=optimization_dataset_yaml,
            device=device,
            all_rows=all_rows,
            promotion_rows=promotion_rows,
            results_csv=results_csv,
            promotions_csv=promotions_csv,
            budget_tracker=budget_tracker,
        )
        require_all_pass(confirmed_rows, f"Generation {generation} full-fidelity confirmation")

        successful_full_rows = [
            row for row in all_rows
            if row["status"] == "PASS"
            and int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
        ]
        incumbent_after = float(rank_mcso_rows(successful_full_rows)[0]["val_map50_95"])
        improvement = incumbent_after - incumbent_before
        previous_generation_improved = improvement > STAGNATION_TOLERANCE

        generation_diversity = mean_pairwise_distance(
            [row_to_vector(row) for row in generation_low_rows]
        )

        stage_summary_rows.append({
            "stage": f"generation_{generation}",
            "generation": generation,
            "mode": mode,
            "step_scale": step_scale,
            "candidate_count_20": len(generation_low_rows),
            "confirmation_count_40": len(confirmed_rows),
            "mean_pairwise_distance_20": generation_diversity,
            "best_map50_95_20": float(rank_mcso_rows(generation_low_rows)[0]["val_map50_95"]),
            "best_confirmed_map50_95_40": float(confirmed_rows[0]["val_map50_95"]),
            "incumbent_map50_95_40": incumbent_after,
            "paired_fidelity_history_size": len(build_paired_fidelity_history(all_rows)),
            "best_predicted_map50_95_40": max(
                float(row["af_predicted_map50_95_40"])
                for row in generation_low_rows
            ),
            "improved_full_fidelity": previous_generation_improved,
        })
        save_rows_to_csv_safe(stage_summary_rows, stage_summary_csv)

        print(f"Incumbent before: {incumbent_before:.6f}")
        print(f"Incumbent after:  {incumbent_after:.6f}")
        print(f"Improvement:      {improvement:+.6f}")

    # ========================================================
    # Exact-budget audit and full-fidelity-only winner
    # ========================================================
    successful_rows = [row for row in all_rows if row["status"] == "PASS"]
    actual_successful_units = sum(int(row["budget_epochs"]) for row in successful_rows)

    if REQUIRE_FULL_HPO_BUDGET:
        if budget_tracker["attempted_epoch_units"] != PLANNED_HPO_EPOCH_UNITS:
            raise RuntimeError(
                f"Expected {PLANNED_HPO_EPOCH_UNITS} attempted HPO epoch-units, "
                f"got {budget_tracker['attempted_epoch_units']}."
            )
        if budget_tracker["attempted_20epoch"] != EXPECTED_LOW_FIDELITY_EVALUATIONS:
            raise RuntimeError("20-epoch evaluation count does not match the frozen plan.")
        if budget_tracker["attempted_40epoch"] != EXPECTED_FULL_FIDELITY_EVALUATIONS:
            raise RuntimeError("40-epoch evaluation count does not match the frozen plan.")
        if len(successful_rows) != EXPECTED_TOTAL_HPO_EVALUATIONS:
            raise RuntimeError(
                f"Expected {EXPECTED_TOTAL_HPO_EVALUATIONS} successful HPO evaluations, "
                f"got {len(successful_rows)}."
            )
        if actual_successful_units != PLANNED_HPO_EPOCH_UNITS:
            raise RuntimeError(
                f"Expected {PLANNED_HPO_EPOCH_UNITS} successful HPO epoch-units, "
                f"got {actual_successful_units}."
            )

    full_rows = [
        row for row in successful_rows
        if int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
    ]
    if len(full_rows) != EXPECTED_FULL_FIDELITY_EVALUATIONS:
        raise RuntimeError(
            f"Expected {EXPECTED_FULL_FIDELITY_EVALUATIONS} successful 40-epoch evaluations, "
            f"got {len(full_rows)}."
        )

    ranked_full = rank_mcso_rows(full_rows)
    best_row = ranked_full[0]

    selected_config = {
        "method": "Adaptive Fidelity-Aware Modified Cuckoo Search (AF-MCSO-320)",
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "selection_rule": (
            "best successful 40-epoch validation mAP50-95; "
            "ties by mAP50, recall, candidate_id"
        ),
        "test_set_used_for_selection": False,
        "selected_candidate_id": best_row["candidate_id"],
        "selected_stage": best_row["stage"],
        "selected_generation": int(best_row["generation"]),
        "selected_role": best_row["role"],
        "selected_validation_map50_95": float(best_row["val_map50_95"]),
        "selected_validation_map50": float(best_row["val_map50"]),
        "selected_validation_recall": float(best_row["val_recall"]),
        "learning_rate": float(best_row["learning_rate"]),
        "lrf": float(best_row["lrf"]),
        "momentum": float(best_row["momentum"]),
        "weight_decay": float(best_row["weight_decay"]),
        "scale": float(best_row["scale"]),
        "mcso_random_seed": int(MCSO_RANDOM_SEED),
        "levy_beta": float(LEVY_BETA),
        "generations": int(MCSO_GENERATIONS),
        "adaptive_fidelity_k_neighbors": int(AF_K_NEIGHBORS),
        "adaptive_fidelity_role": (
            "promotion only; final HPO selection uses actual 40-epoch validation mAP50-95"
        ),
        "planned_hpo_epoch_units": int(PLANNED_HPO_EPOCH_UNITS),
        "attempted_hpo_epoch_units": int(budget_tracker["attempted_epoch_units"]),
        "actual_successful_hpo_epoch_units": int(actual_successful_units),
        "hpo_successful_evaluations": int(len(successful_rows)),
        "hpo_full_fidelity_evaluations": int(len(full_rows)),
        "final_epochs": int(FINAL_EPOCHS),
        "search_space": SEARCH_SPACE,
    }

    print("\n" + "=" * 100)
    print("AF-MCSO HPO COMPLETE")
    print("=" * 100)
    print("Successful HPO evaluations:", len(successful_rows))
    print("20-epoch evaluations:", EXPECTED_LOW_FIDELITY_EVALUATIONS)
    print("40-epoch evaluations:", len(full_rows))
    print("HPO epoch-units:", actual_successful_units)
    print("Selected:", selected_config["selected_candidate_id"])
    print("Selected validation mAP50-95:", selected_config["selected_validation_map50_95"])
    print("Test set used for selection: NO")
    print("=" * 100)

    return selected_config, all_rows, promotion_rows, stage_summary_rows


## Module 9: Compact AF-MCSO progress plot

Creates one search-process figure showing the validation mAP50-95 values of successful 20-epoch and 40-epoch evaluations. This keeps the notebook informative without adding redundant optimizer figures.


In [ ]:
def plot_af_mcso_diagnostics(
    *,
    search_rows: List[Dict[str, Any]],
    promotion_rows: List[Dict[str, Any]],
    stage_summary_rows: List[Dict[str, Any]],
) -> None:
    """Plot successful AF-MCSO validation results by evaluation order and fidelity."""
    del promotion_rows, stage_summary_rows
    RUN_PLOTS_ROOT.mkdir(parents=True, exist_ok=True)

    successful = [row for row in search_rows if row.get("status") == "PASS"]
    if not successful:
        return

    x20, y20, x40, y40 = [], [], [], []
    for index, row in enumerate(successful, start=1):
        budget = int(row["budget_epochs"])
        score = float(row["val_map50_95"])
        if budget == LOW_FIDELITY_EPOCHS:
            x20.append(index)
            y20.append(score)
        elif budget == FULL_FIDELITY_EPOCHS:
            x40.append(index)
            y40.append(score)

    plt.figure(figsize=(8, 5))
    if x20:
        plt.scatter(x20, y20, label="20-epoch evaluations")
    if x40:
        plt.scatter(x40, y40, marker="s", label="40-epoch evaluations")
        incumbent = []
        best = -float("inf")
        for value in y40:
            best = max(best, value)
            incumbent.append(best)
        plt.plot(x40, incumbent, label="40-epoch incumbent")

    plt.xlabel("Successful HPO evaluation order")
    plt.ylabel("Source-validation mAP50-95")
    plt.title(f"{CROSS_YEAR_TAG}: AF-MCSO validation progress")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RUN_PLOTS_ROOT / "af_mcso_source_validation_progress.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()


## Module 10: Isolated target-year test evaluator

Evaluates a frozen final checkpoint on the target-year test split only after HPO selection is complete. Target-test metrics cannot affect prediction, promotion, archive updates, or winner selection.


In [ ]:
def evaluate_checkpoint_on_target_test_split(
    *,
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    expected_class_names: List[str],
    image_size: int,
    batch_size: int,
    workers: int,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on the target-year test split only."""
    checkpoint_path = resolve_repo_path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    output_root.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(checkpoint_path))

    test_start = time.perf_counter()
    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        plots=False,
        save_json=False,
        verbose=False,
    )
    test_seconds = time.perf_counter() - test_start

    metric_data = extract_box_metrics(test_metrics, expected_class_names)

    per_class_rows = [
        {
            "class_id": row["class_id"],
            "class_name": row["class_name"],
            "target_test_map50_95": row["map50_95"],
        }
        for row in metric_data["per_class_map50_95"]
    ]

    return {
        "target_test_precision": metric_data["precision"],
        "target_test_recall": metric_data["recall"],
        "target_test_map50": metric_data["map50"],
        "target_test_map50_95": metric_data["map50_95"],
        "target_test_bottom3_mean_map50_95": metric_data["bottom3_mean_map50_95"],
        "target_test_seconds": float(test_seconds),
        "per_class_rows": per_class_rows,
    }

## Module 11: Complete cross-year workflow

Builds and audits both yearly datasets, prepares one fixed optimization split, runs source-only AF-MCSO HPO, freezes the best measured 40-epoch configuration, then performs final source-year training and isolated target-year testing for seeds 42, 123, and 999.


In [ ]:
def run_cross_year_af_mcso() -> Dict[str, Any]:
    """Run cross-year AF-MCSO HPO, freeze the winner, then perform three-seed target testing."""
    workflow_start = time.perf_counter()
    device = "0" if torch.cuda.is_available() else "cpu"

    ensure_output_directories()

    source_records, source_scan_stats = build_clean_records(TRAIN_SOURCE_ROOT)
    target_records, target_scan_stats = build_clean_records(TEST_TARGET_ROOT)
    source_audit = audit_records(source_records, CLASS_NAMES)
    target_audit = audit_records(target_records, CLASS_NAMES)

    if source_audit["missing_classes"]:
        raise RuntimeError(
            f"Source dataset is missing target classes: {source_audit['missing_classes']}"
        )
    if target_audit["missing_classes"]:
        raise RuntimeError(
            f"Target dataset is missing target classes: {target_audit['missing_classes']}"
        )

    save_json(source_scan_stats, STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_scan_summary.json")
    save_json(target_scan_stats, STATS_ROOT / f"{TEST_DATASET_TAG}_target_scan_summary.json")
    save_json(source_audit, STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_class_audit.json")
    save_json(target_audit, STATS_ROOT / f"{TEST_DATASET_TAG}_target_class_audit.json")
    plot_source_target_box_counts(source_audit, target_audit, RUN_PLOTS_ROOT)

    search_results_csv = RUN_STATS_ROOT / "af_mcso_source_validation_results.csv"
    promotions_csv = RUN_STATS_ROOT / "af_mcso_promotions_20_to_40.csv"
    stage_summary_csv = RUN_STATS_ROOT / "af_mcso_stage_summary.csv"
    selected_json = RUN_STATS_ROOT / "selected_af_mcso_config.json"
    boundary_csv = RUN_STATS_ROOT / "selected_config_boundary_report.csv"

    final_results_csv = RUN_STATS_ROOT / "final_three_run_cross_year_results.csv"
    final_summary_csv = RUN_STATS_ROOT / "final_three_run_mean_sd_summary.csv"
    final_per_class_csv = RUN_STATS_ROOT / "final_three_run_target_test_per_class.csv"
    final_per_class_summary_csv = RUN_STATS_ROOT / "final_three_run_target_test_per_class_mean_sd.csv"
    manifest_json = RUN_STATS_ROOT / "experiment_manifest.json"

    print("=" * 100)
    print("CROSS-YEAR AF-MCSO : COMPETITIVE 320-UNIT HPO")
    print("=" * 100)
    print("Direction:", CROSS_YEAR_TAG)
    print("Optimization split seed:", OPTIMIZATION_SPLIT_SEED)
    print("Optimization training seed:", OPTIMIZATION_TRAIN_SEED)
    print("MCSO seed:", MCSO_RANDOM_SEED)
    print("Objective: SOURCE-year validation mAP50-95 only")
    print("Planned HPO epoch-units:", PLANNED_HPO_EPOCH_UNITS)
    print("Final evaluation epoch-units:", FINAL_EVALUATION_EPOCH_UNITS)
    print("Year2021-source subsampling:", SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE)
    print("Target-year test used during HPO: NO")
    print("Device:", device)
    print("=" * 100)

    # --------------------------------------------------------
    # Stage 1: prepare the single fixed cross-year optimization split.
    # This preparation is excluded from optimizer selection time.
    # --------------------------------------------------------
    optimization_cross, optimization_source, optimization_target, optimization_info = (
        make_cross_year_splits_for_seed(
            OPTIMIZATION_SPLIT_SEED, source_records, target_records
        )
    )

    optimization_dataset_root = YOLO_DATA_ROOT / f"afmcso_opt_seed{OPTIMIZATION_SPLIT_SEED}"
    optimization_manifest_root = (
        SPLITS_ROOT / CROSS_YEAR_TAG / f"afmcso_opt_seed{OPTIMIZATION_SPLIT_SEED}"
    )

    optimization_dataset_yaml, optimization_coverage = prepare_cross_year_yolo_dataset(
        cross_year_splits=optimization_cross,
        source_splits=optimization_source,
        target_splits=optimization_target,
        split_info=optimization_info,
        output_root=optimization_dataset_root,
        manifest_root=optimization_manifest_root,
        image_mode=IMAGE_PLACEMENT_MODE,
        seed=OPTIMIZATION_SPLIT_SEED,
    )

    # --------------------------------------------------------
    # Stage 2: AF-MCSO HPO using source train/validation only.
    # Timing starts after data preparation for comparability.
    # --------------------------------------------------------
    selection_start = time.perf_counter()

    selected_config, search_rows, promotion_rows, stage_summary_rows = run_af_mcso_hpo(
        optimization_dataset_yaml=optimization_dataset_yaml,
        device=device,
        results_csv=search_results_csv,
        promotions_csv=promotions_csv,
        stage_summary_csv=stage_summary_csv,
    )

    selection_stage_wall_seconds = time.perf_counter() - selection_start

    save_json(selected_config, selected_json)
    save_rows_to_csv_safe(build_boundary_report(selected_config), boundary_csv)

    plot_af_mcso_diagnostics(
        search_rows=search_rows,
        promotion_rows=promotion_rows,
        stage_summary_rows=stage_summary_rows,
    )

    # --------------------------------------------------------
    # Stage 3: freeze the HPO winner, then perform three final
    # source-year trainings and isolated target-year tests.
    # --------------------------------------------------------
    final_stage_start = time.perf_counter()

    final_rows: List[Dict[str, Any]] = []
    final_per_class_rows: List[Dict[str, Any]] = []

    for seed in FINAL_EVALUATION_SEEDS:
        print("\n" + "=" * 100)
        print(f"FINAL CROSS-YEAR AF-MCSO RUN : {CROSS_YEAR_TAG} : SEED {seed}")
        print("=" * 100)

        final_cross, final_source, final_target, final_split_info = (
            make_cross_year_splits_for_seed(int(seed), source_records, target_records)
        )

        final_dataset_root = YOLO_DATA_ROOT / f"final_seed{seed}"
        final_manifest_root = SPLITS_ROOT / CROSS_YEAR_TAG / f"final_seed{seed}"

        final_dataset_yaml, final_coverage = prepare_cross_year_yolo_dataset(
            cross_year_splits=final_cross,
            source_splits=final_source,
            target_splits=final_target,
            split_info=final_split_info,
            output_root=final_dataset_root,
            manifest_root=final_manifest_root,
            image_mode=IMAGE_PLACEMENT_MODE,
            seed=int(seed),
        )

        run_start = time.perf_counter()

        final_train = train_and_validate_configuration(
            learning_rate=float(selected_config["learning_rate"]),
            lrf=float(selected_config["lrf"]),
            momentum=float(selected_config["momentum"]),
            weight_decay=float(selected_config["weight_decay"]),
            scale=float(selected_config["scale"]),
            dataset_yaml=final_dataset_yaml,
            project_dir=FINAL_TRAINING_ROOT,
            model_name=MODEL_NAME,
            epochs=FINAL_EPOCHS,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            patience=PATIENCE,
            workers=WORKERS,
            seed=int(seed),
            warmup_epochs=FINAL_WARMUP_EPOCHS,
            close_mosaic=FINAL_CLOSE_MOSAIC,
            fixed_training_args=FIXED_TRAINING_ARGS,
            expected_class_names=CLASS_NAMES,
            strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
            fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
            device=device,
        )

        target_test_result = evaluate_checkpoint_on_target_test_split(
            checkpoint_path=resolve_repo_path(final_train["paths"]["best_checkpoint"]),
            dataset_yaml=final_dataset_yaml,
            output_root=TARGET_TEST_EVAL_ROOT,
            run_name=f"{CROSS_YEAR_TAG}_seed{seed}_target_test",
            expected_class_names=CLASS_NAMES,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            workers=WORKERS,
            device=device,
        )

        run_wall_seconds = time.perf_counter() - run_start
        source_val_metrics = final_train["metrics"]

        final_row = {
            "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR,
            "test_year": TEST_YEAR,
            "seed": int(seed),
            "selected_candidate_id": selected_config["selected_candidate_id"],
            "learning_rate": float(selected_config["learning_rate"]),
            "lrf": float(selected_config["lrf"]),
            "momentum": float(selected_config["momentum"]),
            "weight_decay": float(selected_config["weight_decay"]),
            "scale": float(selected_config["scale"]),
            "source_train_count": int(final_split_info["effective_cross_year_counts"]["train"]),
            "source_valid_count": int(final_split_info["effective_cross_year_counts"]["valid"]),
            "target_test_count": int(final_split_info["effective_cross_year_counts"]["test"]),
            "subsample_enabled": bool(final_split_info["subsample_info"]["enabled"]),
            "subsample_seed": final_split_info["subsample_info"]["subsample_seed"],
            "final_source_val_precision": float(source_val_metrics["precision"]),
            "final_source_val_recall": float(source_val_metrics["recall"]),
            "final_source_val_map50": float(source_val_metrics["map50"]),
            "final_source_val_map50_95": float(source_val_metrics["map50_95"]),
            "final_source_val_bottom3_mean_map50_95": float(source_val_metrics["bottom3_mean_map50_95"]),
            "target_test_precision": float(target_test_result["target_test_precision"]),
            "target_test_recall": float(target_test_result["target_test_recall"]),
            "target_test_map50": float(target_test_result["target_test_map50"]),
            "target_test_map50_95": float(target_test_result["target_test_map50_95"]),
            "target_test_bottom3_mean_map50_95": float(target_test_result["target_test_bottom3_mean_map50_95"]),
            "final_training_seconds": float(final_train["timing"]["training_seconds"]),
            "final_source_validation_seconds": float(final_train["timing"]["validation_seconds"]),
            "target_test_seconds": float(target_test_result["target_test_seconds"]),
            "final_run_wall_seconds": float(run_wall_seconds),
            "best_checkpoint": final_train["paths"]["best_checkpoint"],
        }
        final_rows.append(final_row)

        for row in target_test_result["per_class_rows"]:
            final_per_class_rows.append({
                "seed": int(seed),
                **row,
            })

        save_rows_to_csv_safe(final_rows, final_results_csv)
        save_rows_to_csv_safe(final_per_class_rows, final_per_class_csv)

    if REQUIRE_ALL_FINAL_RUNS and len(final_rows) != len(FINAL_EVALUATION_SEEDS):
        raise RuntimeError(
            f"Expected {len(FINAL_EVALUATION_SEEDS)} final runs, got {len(final_rows)}."
        )

    final_stage_wall_seconds = time.perf_counter() - final_stage_start
    total_workflow_wall_seconds = time.perf_counter() - workflow_start

    successful_hpo = [row for row in search_rows if row["status"] == "PASS"]
    successful_hpo_units = sum(int(row["budget_epochs"]) for row in successful_hpo)
    full_hpo_rows = [
        row for row in successful_hpo
        if int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
    ]

    summary_metrics = [
        "final_source_val_precision",
        "final_source_val_recall",
        "final_source_val_map50",
        "final_source_val_map50_95",
        "final_source_val_bottom3_mean_map50_95",
        "target_test_precision",
        "target_test_recall",
        "target_test_map50",
        "target_test_map50_95",
        "target_test_bottom3_mean_map50_95",
        "final_training_seconds",
        "final_source_validation_seconds",
        "target_test_seconds",
        "final_run_wall_seconds",
    ]
    summary_rows = summarize_numeric_rows(final_rows, summary_metrics)

    paired_history_count = len(build_paired_fidelity_history(search_rows))
    summary_rows.extend([
        {
            "metric": "hpo_epoch_units", "n": 1,
            "mean": int(PLANNED_HPO_EPOCH_UNITS), "sd": "",
            "min": int(PLANNED_HPO_EPOCH_UNITS), "max": int(PLANNED_HPO_EPOCH_UNITS),
        },
        {
            "metric": "actual_successful_hpo_epoch_units", "n": 1,
            "mean": int(successful_hpo_units), "sd": "",
            "min": int(successful_hpo_units), "max": int(successful_hpo_units),
        },
        {
            "metric": "hpo_successful_evaluations", "n": 1,
            "mean": int(len(successful_hpo)), "sd": "",
            "min": int(len(successful_hpo)), "max": int(len(successful_hpo)),
        },
        {
            "metric": "hpo_full_fidelity_evaluations", "n": 1,
            "mean": int(len(full_hpo_rows)), "sd": "",
            "min": int(len(full_hpo_rows)), "max": int(len(full_hpo_rows)),
        },
        {
            "metric": "final_fidelity_history_pairs", "n": 1,
            "mean": int(paired_history_count), "sd": "",
            "min": int(paired_history_count), "max": int(paired_history_count),
        },
        {
            "metric": "final_evaluation_epoch_units", "n": 1,
            "mean": int(FINAL_EVALUATION_EPOCH_UNITS), "sd": "",
            "min": int(FINAL_EVALUATION_EPOCH_UNITS), "max": int(FINAL_EVALUATION_EPOCH_UNITS),
        },
        {
            "metric": "selection_stage_wall_seconds", "n": 1,
            "mean": float(selection_stage_wall_seconds), "sd": "",
            "min": float(selection_stage_wall_seconds), "max": float(selection_stage_wall_seconds),
        },
        {
            "metric": "final_stage_wall_seconds", "n": 1,
            "mean": float(final_stage_wall_seconds), "sd": "",
            "min": float(final_stage_wall_seconds), "max": float(final_stage_wall_seconds),
        },
        {
            "metric": "total_workflow_wall_seconds", "n": 1,
            "mean": float(total_workflow_wall_seconds), "sd": "",
            "min": float(total_workflow_wall_seconds), "max": float(total_workflow_wall_seconds),
        },
    ])
    save_rows_to_csv_safe(summary_rows, final_summary_csv)

    per_class_summary_rows = summarize_per_class_rows(final_per_class_rows)
    save_rows_to_csv_safe(per_class_summary_rows, final_per_class_summary_csv)

    # Compact final target-test metric summary.
    plot_metrics = [
        ("Precision", "target_test_precision"),
        ("Recall", "target_test_recall"),
        ("mAP50", "target_test_map50"),
        ("mAP50-95", "target_test_map50_95"),
    ]
    plot_means = []
    plot_sds = []
    for _, key in plot_metrics:
        values = [float(row[key]) for row in final_rows]
        plot_means.append(float(mean(values)))
        plot_sds.append(float(stdev(values)) if len(values) > 1 else 0.0)

    plt.figure(figsize=(8, 5))
    plt.bar(
        [label for label, _ in plot_metrics],
        plot_means,
        yerr=plot_sds,
        capsize=5,
    )
    plt.ylabel("Target test metric")
    plt.title(f"{CROSS_YEAR_TAG}: AF-MCSO target test mean +/- sample SD")
    plt.ylim(0.0, 1.0)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        RUN_PLOTS_ROOT / "af_mcso_target_test_mean_sd.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()

    manifest = {
        "method": "Cross-Year Adaptive Fidelity-Aware Modified Cuckoo Search (AF-MCSO-320)",
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "source_dataset_root": portable_path(TRAIN_SOURCE_ROOT),
        "target_dataset_root": portable_path(TEST_TARGET_ROOT),
        "source_dataset_scan_summary": source_scan_stats,
        "target_dataset_scan_summary": target_scan_stats,
        "source_dataset_class_audit": source_audit,
        "target_dataset_class_audit": target_audit,
        "objective": "source-year validation mAP50-95 only",
        "target_test_used_during_hpo": False,
        "search_space_version": SEARCH_SPACE_VERSION,
        "search_space": SEARCH_SPACE,
        "fixed_training_args": FIXED_TRAINING_ARGS,
        "runtime": {
            "model_name": MODEL_NAME,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "workers": WORKERS,
            "cache": CACHE_MODE,
            "patience": PATIENCE,
            "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
            "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
            "final_evaluation_seeds": FINAL_EVALUATION_SEEDS,
        },
        "cross_year_protocol": {
            "effective_train": "source-year train split",
            "effective_validation": "source-year validation split",
            "effective_test": "target-year test split",
            "optimization_split_info": optimization_info,
            "year2021_source_subsampling": {
                "enabled_if_source_2021": SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE,
                "mode": SUBSAMPLE_TARGET_MODE,
                "seed_offset": SUBSAMPLE_SEED_OFFSET,
                "description": (
                    "only the Year2021 source training split is reduced to the target-year "
                    "training-split size using image-level primary-label stratification"
                ),
            },
        },
        "fidelity": {
            "low_epochs": LOW_FIDELITY_EPOCHS,
            "low_warmup_epochs": LOW_FIDELITY_WARMUP_EPOCHS,
            "low_close_mosaic": LOW_FIDELITY_CLOSE_MOSAIC,
            "full_epochs": FULL_FIDELITY_EPOCHS,
            "full_warmup_epochs": FULL_FIDELITY_WARMUP_EPOCHS,
            "full_close_mosaic": FULL_FIDELITY_CLOSE_MOSAIC,
        },
        "af_mcso": {
            "mcso_random_seed": MCSO_RANDOM_SEED,
            "initial_nests": INITIAL_NESTS,
            "initial_confirmations": INITIAL_CONFIRMATIONS,
            "generations": MCSO_GENERATIONS,
            "candidates_per_generation": CANDIDATES_PER_GENERATION,
            "promotions_per_generation": PROMOTIONS_PER_GENERATION,
            "levy_beta": LEVY_BETA,
            "generation_step_scales": GENERATION_STEP_SCALES,
            "af_k_neighbors": AF_K_NEIGHBORS,
            "af_min_history_pairs": AF_MIN_HISTORY_PAIRS,
            "af_prediction": (
                "inverse-distance weighted 20->40 source-validation mAP50-95 gain "
                "in normalized 5D space"
            ),
            "promotion_rule": "highest predicted 40-epoch source-validation mAP50-95",
            "final_selection_rule": "actual 40-epoch source-validation mAP50-95 only",
        },
        "budget": {
            "hpo_epoch_units": PLANNED_HPO_EPOCH_UNITS,
            "expected_low_fidelity_evaluations": EXPECTED_LOW_FIDELITY_EVALUATIONS,
            "expected_full_fidelity_evaluations": EXPECTED_FULL_FIDELITY_EVALUATIONS,
            "expected_total_hpo_evaluations": EXPECTED_TOTAL_HPO_EVALUATIONS,
            "final_evaluation_epoch_units": FINAL_EVALUATION_EPOCH_UNITS,
            "full_workflow_epoch_units": PLANNED_FULL_WORKFLOW_EPOCH_UNITS,
        },
        "selected_config": selected_config,
        "timing": {
            "selection_stage_wall_seconds": float(selection_stage_wall_seconds),
            "final_stage_wall_seconds": float(final_stage_wall_seconds),
            "total_workflow_wall_seconds": float(total_workflow_wall_seconds),
        },
        "software": {
            "python": sys.version,
            "ultralytics": ultralytics.__version__,
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
    }
    save_json(manifest, manifest_json)

    print("\n" + "=" * 100)
    print("CROSS-YEAR AF-MCSO COMPLETE")
    print("=" * 100)
    print("Direction:", CROSS_YEAR_TAG)
    print("Selected candidate:", selected_config["selected_candidate_id"])
    print("Selected 40-epoch SOURCE-validation mAP50-95:", selected_config["selected_validation_map50_95"])
    print("HPO epoch-units:", PLANNED_HPO_EPOCH_UNITS)
    print("Successful HPO epoch-units:", successful_hpo_units)
    print("Selection time (s):", selection_stage_wall_seconds)
    print("Final stage time (s):", final_stage_wall_seconds)
    print("Total workflow time (s):", total_workflow_wall_seconds)
    print("\nFinal mean +/- SD:")
    for row in summary_rows:
        if row["metric"] in {
            "final_source_val_precision",
            "final_source_val_recall",
            "final_source_val_map50",
            "final_source_val_map50_95",
            "final_source_val_bottom3_mean_map50_95",
            "target_test_precision",
            "target_test_recall",
            "target_test_map50",
            "target_test_map50_95",
            "target_test_bottom3_mean_map50_95",
        }:
            sd_text = f"{float(row['sd']):.6f}" if row["sd"] != "" else "NA"
            print(f"{row['metric']}: {float(row['mean']):.6f} +/- {sd_text}")
    print("=" * 100)

    return {
        "selected_config": selected_config,
        "search_rows": search_rows,
        "promotion_rows": promotion_rows,
        "stage_summary_rows": stage_summary_rows,
        "final_rows": final_rows,
        "summary_rows": summary_rows,
        "manifest": manifest,
    }


## Module 12: Run switch and post-success cleanup

Keep the switch `False` for GitHub review. Set it to `True` to run the complete experiment. After every required HPO and final evaluation step succeeds, the large HPO training folder and the complete `yolo_data` folder are deleted. Statistics, manifests, plots, and final training outputs are retained.


In [ ]:
# Keep False while reviewing the notebook on GitHub.
# Set True only when ready to run the complete experiment.
RUN_CROSS_YEAR_AF_MCSO = False

if RUN_CROSS_YEAR_AF_MCSO:
    cross_year_af_mcso_result = run_cross_year_af_mcso()

    # Remove large temporary folders only after the complete workflow succeeds.
    if HPO_RUNS_ROOT.exists():
        shutil.rmtree(HPO_RUNS_ROOT)
        print("Deleted temporary HPO training folder:", portable_path(HPO_RUNS_ROOT))

    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(YOLO_DATA_ROOT)
        print("Deleted temporary YOLO data folder:", portable_path(YOLO_DATA_ROOT))
else:
    print("RUN_CROSS_YEAR_AF_MCSO is False.")
    print("Direction:", CROSS_YEAR_TAG)
    print("Change only TRAIN_YEAR and TEST_YEAR to reverse the experiment.")
    print("Objective: source-year validation mAP50-95 only")
    print("AF-MCSO fidelity:", LOW_FIDELITY_EPOCHS, "to", FULL_FIDELITY_EPOCHS)
    print("Planned HPO epoch-units:", PLANNED_HPO_EPOCH_UNITS)
    print("Expected HPO evaluations:", EXPECTED_TOTAL_HPO_EVALUATIONS)
    print("Expected 20-epoch evaluations:", EXPECTED_LOW_FIDELITY_EVALUATIONS)
    print("Expected 40-epoch evaluations:", EXPECTED_FULL_FIDELITY_EVALUATIONS)
    print("Final evaluation epoch-units:", FINAL_EVALUATION_EPOCH_UNITS)
    print("Year2021 source-training subsampling:", SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE)
    print("Only measured 40-epoch source-validation mAP50-95 can determine the HPO winner.")
    print("The target-year test split is never used during HPO.")


## Frozen cross-year AF-MCSO safeguards

- Change only `TRAIN_YEAR` and `TEST_YEAR` to reverse the cross-year direction. The two years must differ.
- Tuned parameters are `lr0`, `lrf`, `momentum`, `weight_decay`, and `scale`.
- HPO uses source-year training and source-year validation only.
- The target-year test split is isolated until the HPO winner is frozen.
- When Year2021 is the source year, only its training split is reduced to the target-year training-split size.
- HPO split seed is 42. HPO training seed is 42. AF-MCSO random seed is 9187.
- Final evaluation seeds are 42, 123, and 999.
- Fidelities are 20 epochs with `warmup=1.5`, `close_mosaic=5`, and 40 epochs with `warmup=3.0`, `close_mosaic=10`.
- Early stopping is disabled with `patience=0`.
- Adaptive-fidelity prediction uses only completed 20-to-40 epoch source-validation pairs.
- The HPO winner is selected only from measured 40-epoch source-validation mAP50-95.
- HPO budget is exactly 320 epoch-units. Final three-seed evaluation adds 120 epoch-units and is reported separately.
- Failed scheduled HPO evaluations are not silently replaced.
- Split manifests contain filenames only.
- The HPO training folder and `YOLO_DATA_ROOT` are removed only after successful completion.
